# 01. Limpieza e integración de datos

**Notebook anterior:** `00_inicializacion_datos_movielens.ipynb`

### Introducción a la etapa de limpieza e integración

Después de completar la adquisición, validación y comprensión inicial del conjunto de datos MovieLens 32M, se inicia la primera fase de preparación de datos orientada al análisis.

Durante la etapa anterior se verificó la integridad de los archivos descargados, se inspeccionó la estructura general de los datos y se realizó una caracterización preliminar de las tablas principales. Sin embargo, antes de comenzar cualquier análisis estadístico o exploratorio avanzado, resulta necesario evaluar la calidad de los datos, comprender las relaciones existentes entre las entidades del modelo y definir una estrategia de integración que preserve la semántica original de la información.

Esta fase constituye el puente entre la adquisición de datos y el análisis exploratorio formal, asegurando que las decisiones posteriores se realicen sobre estructuras limpias, documentadas y reproducibles.

### Objetivo de la preparación del entorno

Antes de iniciar las actividades de limpieza e integración, es necesario configurar nuevamente el entorno de trabajo.

Aunque los datos ya fueron cargados e inspeccionados en el notebook anterior, este proceso se diseña bajo criterios de reproducibilidad. Por esta razón, el notebook vuelve a establecer todas las dependencias, rutas y configuraciones necesarias para funcionar de manera autónoma.

De esta forma:

- No depende de variables previamente almacenadas en memoria.
- Puede ejecutarse de forma independiente por cualquier integrante del equipo.
- Garantiza resultados consistentes entre distintas sesiones de trabajo.
- Facilita la replicación completa del proyecto por terceros.

### Importación de bibliotecas

En esta etapa se incorporan bibliotecas orientadas tanto a la manipulación de datos como a la gestión de archivos y diagnósticos de calidad.

#### Manipulación y análisis de datos

- **Pandas (`pandas`)**: será la biblioteca principal para limpieza, transformación e integración de tablas.
- **NumPy (`numpy`)**: proporcionará soporte para operaciones numéricas y manejo eficiente de estructuras matriciales.

#### Gestión de directorios y archivos

- **Pathlib (`Path`)**: permite construir rutas independientes del sistema operativo.
- **Shutil (`shutil`)**: incorpora utilidades para copias y operaciones sobre archivos y directorios.

#### Diagnóstico y trazabilidad

- **JSON (`json`)**: será utilizado para almacenar resultados de validaciones y diagnósticos.
- **Hashlib (`hashlib`)**: permitirá generar identificadores hash para comprobaciones de integridad y trazabilidad.
- **Warnings (`warnings`)**: facilita la gestión controlada de advertencias durante el procesamiento.

Estas dependencias conforman la base sobre la cual se desarrollarán los análisis de calidad de datos y los procesos de integración posteriores.

### Configuración del entorno de inspección

Adicionalmente, se ajustan algunas opciones de visualización de Pandas para facilitar la exploración de los datos.

Estas configuraciones permiten:

- Visualizar un mayor número de columnas simultáneamente.
- Inspeccionar más registros durante las etapas de diagnóstico.
- Presentar valores numéricos utilizando formatos homogéneos.

El objetivo es mejorar la legibilidad de los resultados y facilitar la interpretación de los diagnósticos que se generarán a lo largo del proceso.

### Detección automática de la raíz del proyecto

Uno de los principios de este proyecto es minimizar las dependencias respecto a la ubicación desde la cual se ejecuta el notebook.

Por esta razón, en lugar de asumir una ruta fija, se realiza una búsqueda ascendente a partir del directorio de trabajo actual hasta localizar una carpeta que contenga simultáneamente:

- El directorio `data`
- El directorio `notebooks`

Esta validación permite identificar de manera fiable la raíz del proyecto, independientemente del entorno de ejecución utilizado por cada integrante del equipo.

Si dicha estructura no puede localizarse, el proceso se detiene mediante una excepción controlada, evitando que operaciones posteriores se ejecuten sobre rutas incorrectas.

### Definición de directorios de trabajo

Una vez identificada la raíz del proyecto, se configuran los principales directorios que participarán en esta fase:

#### Directorio de datos originales

```text
data/raw/ml-32m/
```

Contiene los archivos originales obtenidos durante la fase de adquisición y que permanecerán inalterados durante todo el proyecto.

Estos archivos se consideran la fuente única de verdad y constituyen el punto de partida para cualquier reprocesamiento futuro.

#### Directorio de datos procesados

```text
data/processed/
```

Almacenará los resultados derivados de las actividades de limpieza, normalización e integración.

Entre los productos que se espera generar se encuentran:

- Tablas depuradas.
- Catálogos enriquecidos.
- Tablas puente.
- Tablas analíticas.
- Versiones derivadas utilizadas en análisis posteriores.

#### Directorio de diagnósticos

```text
data/processed/diagnostics/
```

Se destina al almacenamiento de métricas, reportes y resultados de control de calidad generados durante el proceso.

La separación explícita de estos diagnósticos facilita la trazabilidad de las decisiones tomadas durante la preparación de los datos.

### Garantía de reproducibilidad

Antes de continuar con la carga y validación de las tablas, el notebook verifica automáticamente la existencia de las carpetas de salida necesarias para almacenar los resultados generados en esta fase.

Si alguno de los directorios no existe, éste se crea automáticamente sin afectar la estructura existente del proyecto.

Esta estrategia garantiza que:

- El notebook pueda ejecutarse desde cero.
- Los resultados se almacenen en ubicaciones consistentes.
- Todos los integrantes del equipo compartan la misma organización de archivos.
- El flujo de trabajo sea completamente reproducible.

### Resultado esperado de esta etapa preliminar

Al finalizar esta fase de preparación del entorno, se dispone de una configuración estable y reproducible para iniciar los procedimientos de limpieza e integración.

Con ello se garantiza que las siguientes actividades:

- Evaluación de calidad de datos.
- Detección de inconsistencias.
- Identificación de valores faltantes.
- Análisis de duplicados.
- Normalización de variables.
- Construcción de tablas puente.
- Integración controlada de entidades.

se desarrollen sobre una estructura de trabajo homogénea, trazable y alineada con los principios metodológicos definidos para el proyecto.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import shutil
import warnings

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Localizar la raíz del proyecto.
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "data").exists() and (p / "notebooks").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se localizó la raíz del proyecto. Ejecuta el notebook desde la carpeta "
        "'notebooks' o ajusta PROJECT_ROOT manualmente."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "ml-32m"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DIAGNOSTICS_DIR = PROCESSED_DIR / "diagnostics"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Datos originales: {RAW_DIR}")
print(f"Datos procesados: {PROCESSED_DIR}")

Raíz del proyecto: c:\Users\igpea\OneDrive\Documents\Diplomado\Proyecto_Integrador_M1
Datos originales: c:\Users\igpea\OneDrive\Documents\Diplomado\Proyecto_Integrador_M1\data\raw\ml-32m
Datos procesados: c:\Users\igpea\OneDrive\Documents\Diplomado\Proyecto_Integrador_M1\data\processed


## Carga controlada de las tablas base del modelo relacional

Una vez configurado el entorno de trabajo y establecidas las rutas principales del proyecto, se procede a la carga controlada de las cuatro tablas que conforman el núcleo del conjunto de datos MovieLens 32M.

Esta etapa tiene como objetivo incorporar los datos al entorno de análisis preservando su estructura original, manteniendo la consistencia de los tipos de datos y optimizando el consumo de memoria desde el inicio del proceso.

A diferencia de la exploración preliminar realizada en el notebook anterior, esta carga constituye el punto de partida formal para las actividades de limpieza, validación e integración que se desarrollarán a lo largo de la presente etapa.

## Verificación de disponibilidad de archivos

Antes de leer cualquier conjunto de datos, se construye un diccionario que centraliza las rutas de acceso a los archivos originales:

- `movies.csv`
- `ratings.csv`
- `tags.csv`
- `links.csv`

Posteriormente se realiza una validación preventiva que confirma la existencia física de cada archivo dentro del directorio `data/raw/ml-32m/`.

Esta comprobación tiene como finalidad:

- Detectar archivos faltantes antes de iniciar el procesamiento.
- Evitar errores posteriores durante la lectura.
- Proporcionar mensajes de diagnóstico claros en caso de problemas de configuración.
- Garantizar la reproducibilidad del flujo de trabajo.

Si alguno de los archivos no está disponible, el proceso se interrumpe inmediatamente mediante una excepción controlada (`FileNotFoundError`), evitando que se generen resultados parciales o inconsistentes.

## Comprensión de la estructura relacional del dataset

Antes de realizar cualquier transformación resulta fundamental comprender la unidad de observación representada por cada tabla.

Aunque las cuatro tablas se encuentran relacionadas mediante identificadores comunes, cada una describe fenómenos distintos y opera sobre niveles de granularidad diferentes.

| Tabla | Unidad de observación |
|---------|---------|
| `movies` | Una película del catálogo |
| `ratings` | Una calificación emitida por un usuario |
| `tags` | Una etiqueta asignada por un usuario |
| `links` | Un conjunto de identificadores externos asociados a una película |

Esta distinción resulta particularmente importante durante los procesos de integración posteriores, ya que evita interpretar incorrectamente registros repetidos como duplicados o realizar uniones que multipliquen artificialmente el número de observaciones.

## Carga optimizada de las tablas

Cada archivo es cargado utilizando tipos de datos explícitamente definidos mediante el parámetro `dtype`.

La especificación manual de tipos aporta diversos beneficios:

### Reducción del consumo de memoria

El dataset MovieLens 32M contiene decenas de millones de registros, particularmente en la tabla de calificaciones.

La utilización de tipos compactos como:

- `int32`
- `float32`
- `Int64`
- `string`

reduce significativamente la memoria requerida respecto a los tipos inferidos automáticamente por Pandas.

### Consistencia entre tablas

Los identificadores compartidos entre distintos archivos conservan exactamente el mismo tipo de dato.

Por ejemplo:

- `movieId` se mantiene como `int32` en todas las tablas.
- `userId` conserva el mismo formato tanto en `ratings` como en `tags`.

Esto facilita los procesos de validación e integración posteriores.

### Preparación para operaciones analíticas

La tipificación temprana reduce la necesidad de conversiones posteriores y minimiza el riesgo de errores derivados de inconsistencias entre tablas relacionadas.

## Consideraciones especiales sobre la tabla ratings

Entre las cuatro fuentes de información, la tabla `ratings` representa con diferencia el componente de mayor tamaño del proyecto.

Cada registro de esta tabla corresponde a una interacción entre un usuario y una película, lo que genera un volumen cercano a los 32 millones de observaciones.

Como consecuencia:

- Se utilizan tipos compactos para minimizar el uso de memoria.
- Algunas operaciones posteriores se realizarán mediante procesamiento por bloques.
- Se evitarán copias innecesarias de la tabla durante las transformaciones.

Estas decisiones permiten mantener la eficiencia computacional del proyecto sin comprometer la calidad del análisis.

## Conservación de los timestamps originales

Tanto la tabla `ratings` como la tabla `tags` contienen la variable `timestamp`, almacenada originalmente como un entero Unix.

A diferencia de lo realizado durante la exploración inicial del notebook anterior, en esta fase los valores temporales se mantienen en su forma original durante la carga.

Esta decisión responde a varias razones metodológicas:

- Preservar los datos tal como fueron distribuidos por la fuente original.
- Mantener trazabilidad sobre el valor almacenado en los archivos fuente.
- Evitar transformaciones prematuras.
- Permitir la generación de múltiples representaciones temporales según las necesidades analíticas.

Las conversiones a fechas legibles se realizarán posteriormente sobre columnas auxiliares o dentro de análisis específicos de carácter temporal.

De esta forma se conserva tanto la información original como la capacidad de generar variables derivadas cuando sean necesarias.

## Validación de la carga

Una vez leídos los cuatro archivos, se realiza una verificación básica mostrando para cada tabla:

- Número total de filas.
- Número total de columnas.

Esta comprobación cumple dos objetivos principales:

### Confirmar la lectura correcta de los archivos

Permite verificar que todos los conjuntos de datos fueron cargados sin errores y que el volumen de registros es consistente con lo esperado para la versión MovieLens 32M.

### Dimensionar el proceso de limpieza

Conocer el tamaño relativo de cada tabla facilita la planificación de las tareas posteriores de validación, integración y enriquecimiento.

Especialmente en el caso de `ratings`, la magnitud de los datos influirá directamente sobre las estrategias de procesamiento que se aplicarán en las siguientes etapas.

## Resultado de esta fase

Al finalizar este procedimiento se dispone de las cuatro tablas fundamentales del ecosistema MovieLens correctamente cargadas, tipificadas y listas para iniciar los diagnósticos de calidad de datos.

Además, se garantiza que:

- Todos los archivos requeridos se encuentran disponibles.
- Los tipos de datos son consistentes entre tablas relacionadas.
- Los identificadores mantienen compatibilidad para las futuras integraciones.
- Los timestamps originales permanecen intactos.
- El consumo de memoria se encuentra optimizado para trabajar con volúmenes elevados de información.

Con esta base preparada, el siguiente paso consistirá en evaluar sistemáticamente la calidad de los datos antes de proceder con los procesos de limpieza, normalización e integración.

In [2]:
FILES = {
    "movies": RAW_DIR / "movies.csv",
    "ratings": RAW_DIR / "ratings.csv",
    "tags": RAW_DIR / "tags.csv",
    "links": RAW_DIR / "links.csv",
}

missing_files = [name for name, path in FILES.items() if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"No se encontraron estos archivos: {missing_files}")

movies = pd.read_csv(
    FILES["movies"],
    encoding="utf-8",
    dtype={"movieId": "int32", "title": "string", "genres": "string"}
)

ratings = pd.read_csv(
    FILES["ratings"],
    encoding="utf-8",
    dtype={"userId": "int32", "movieId": "int32",
           "rating": "float32", "timestamp": "int64"}
)

tags = pd.read_csv(
    FILES["tags"],
    encoding="utf-8",
    dtype={"userId": "int32", "movieId": "int32",
           "tag": "string", "timestamp": "int64"}
)

links = pd.read_csv(
    FILES["links"],
    encoding="utf-8",
    dtype={"movieId": "int32", "imdbId": "Int64", "tmdbId": "Int64"}
)

for name, df in {
    "movies": movies, "ratings": ratings, "tags": tags, "links": links
}.items():
    print(f"{name:>8}: {len(df):>12,} filas | {len(df.columns)} columnas")

  movies:       87,585 filas | 3 columnas
 ratings:   32,000,204 filas | 4 columnas
    tags:    2,000,072 filas | 4 columnas
   links:       87,585 filas | 3 columnas


## Validación de estructura y evaluación general de calidad de datos

Antes de realizar cualquier transformación, limpieza o integración entre tablas, resulta fundamental construir un diagnóstico objetivo del estado actual de los datos. Esta evaluación inicial permite establecer una línea base de calidad y documentar las condiciones en las que fueron recibidos los conjuntos de datos originales.

En esta etapa no se modifica ninguna observación ni se realiza ningún tratamiento sobre los registros. El propósito es exclusivamente descriptivo y busca comprender la estructura interna de cada tabla, identificar posibles problemas y reunir evidencia para justificar las decisiones que se tomarán en las fases posteriores de limpieza.

## Construcción de un diagnóstico homogéneo

Dado que el conjunto MovieLens está compuesto por tablas con distintas unidades de observación, se adopta un marco de evaluación común que permita comparar su calidad estructural sin perder de vista las diferencias entre ellas.

Para cada dataset se calculan indicadores relacionados con:

- Volumen de información.
- Completitud de los registros.
- Presencia de valores faltantes.
- Existencia de duplicados completos.
- Diversidad de valores observados.
- Consumo de memoria.

Posteriormente, se realiza un análisis más detallado a nivel de columna para identificar características particulares de cada variable.

## Diferenciación entre conceptos de calidad

Durante esta evaluación es importante distinguir varios conceptos que suelen confundirse durante el análisis exploratorio.

### Registros

Representan el número total de filas contenidas en una tabla.

El volumen de registros permite dimensionar el tamaño efectivo de cada conjunto de datos y entender la granularidad de la información almacenada.

Por ejemplo:

- En `movies`, una fila representa una película.
- En `ratings`, una fila representa una calificación emitida por un usuario.
- En `tags`, una fila corresponde a una etiqueta asignada por un usuario.
- En `links`, una fila representa la correspondencia entre una película y sistemas externos de identificación.

### Valores únicos

La cardinalidad de una variable corresponde al número de valores distintos observados en dicha columna.

Es importante señalar que una cardinalidad baja no implica necesariamente un problema de calidad.

Por ejemplo:

- La variable `rating` únicamente puede tomar un conjunto limitado de valores dentro de la escala definida por MovieLens.
- Variables categóricas como géneros o etiquetas pueden tener cardinalidades muy diferentes debido a su naturaleza.

Por esta razón, los valores únicos se utilizan para describir la diversidad de una variable, pero no deben interpretarse automáticamente como evidencia de registros duplicados.

### Duplicados completos

Los duplicados completos corresponden a filas idénticas en todas sus columnas.

Este indicador permite detectar posibles problemas de captura, carga o almacenamiento de la información.

La evaluación se realiza considerando simultáneamente todas las variables de cada fila, evitando confundir observaciones legítimamente repetidas con duplicados reales.

### Valores nulos

Los valores nulos representan la ausencia de información en uno o más atributos.

La identificación temprana de valores faltantes es fundamental porque puede afectar:

- Procesos de integración.
- Análisis descriptivos.
- Construcción de indicadores.
- Modelos estadísticos o predictivos.

No obstante, la existencia de valores nulos no implica necesariamente un error. En ciertas tablas, algunos atributos externos pueden ser opcionales o simplemente no encontrarse disponibles para determinadas observaciones.

## Evaluación de completitud

Además del conteo de nulos, se calcula el porcentaje de completitud de cada dataset.

La completitud representa la proporción de celdas que contienen información válida respecto al total de valores posibles de una tabla.

Este indicador proporciona una visión más general de la calidad de los datos y facilita la comparación entre datasets de tamaños muy diferentes.

Se utiliza un nivel de precisión elevado durante el cálculo para evitar interpretaciones erróneas. En conjuntos de datos masivos, una proporción reducida de valores faltantes puede aparecer visualmente como `0.00 %` cuando se utilizan pocos decimales, ocultando la presencia real de observaciones incompletas.

## Diagnóstico general por tabla

Para cada conjunto de datos se construye un resumen ejecutivo que incluye:

### Número de registros

Permite dimensionar el volumen total de información disponible.

### Número de columnas

Facilita la comprensión de la complejidad estructural de la tabla.

### Suma de valores únicos

Proporciona una medida agregada de diversidad de información presente en el conjunto de variables.

### Duplicados completos

Cuantifica la existencia de registros idénticos.

### Celdas nulas

Reporta el número total de valores faltantes presentes en la tabla.

### Porcentaje de completitud

Evalúa la proporción de información efectivamente disponible.

### Consumo de memoria

Indica la cantidad de memoria requerida por cada tabla una vez cargada en Pandas.

Este indicador resulta particularmente relevante para tablas de gran volumen como `ratings`, donde las decisiones de tipificación tienen un impacto directo en la eficiencia computacional.

## Diagnóstico a nivel de variable

Posteriormente se realiza una inspección detallada para cada columna de cada tabla.

Este análisis permite construir un perfil de calidad individual para cada atributo.

Las métricas evaluadas incluyen:

### Tipo de dato

Permite verificar que las variables hayan sido interpretadas correctamente durante la carga.

Esta comprobación es especialmente importante para:

- Identificadores.
- Variables numéricas.
- Variables categóricas.
- Variables temporales.

### Valores no nulos

Indican el número de observaciones con información disponible.

### Valores nulos

Permiten identificar atributos con posibles problemas de completitud.

### Porcentaje de nulos

Facilita la comparación entre variables pertenecientes a tablas con tamaños diferentes.

### Cardinalidad

Representa el número de valores distintos observados en cada columna.

Este indicador resulta útil para:

- Detectar posibles llaves candidatas.
- Identificar variables categóricas.
- Evaluar niveles de diversidad.
- Anticipar futuras estrategias de modelado y análisis.

## Importancia metodológica de esta etapa

La validación realizada en esta fase constituye la base para todas las decisiones posteriores de limpieza e integración.

Gracias a este diagnóstico será posible:

- Identificar problemas reales de calidad de datos.
- Diferenciar entre características esperadas y anomalías.
- Justificar tratamientos sobre valores faltantes.
- Evaluar la existencia de duplicados.
- Determinar la necesidad de transformaciones adicionales.
- Documentar el estado original de los datos antes de cualquier modificación.

De esta manera, las siguientes etapas de limpieza y normalización se apoyarán en evidencia cuantitativa objetiva y no en supuestos sobre la calidad de los datos recibidos.

In [3]:
tables = {
    "movies": movies,
    "ratings": ratings,
    "tags": tags,
    "links": links
}

overview_rows = []
column_diagnostics = []

for name, df in tables.items():
    total_cells = df.shape[0] * df.shape[1]
    null_cells = int(df.isna().sum().sum())
    completeness = 100 * (1 - null_cells / total_cells) if total_cells else np.nan

    overview_rows.append({
        "dataset": name,
        "records": len(df),
        "columns": len(df.columns),
        "distinct_values_sum": int(df.nunique(dropna=True).sum()),
        "full_row_duplicates": int(df.duplicated().sum()),
        "null_cells": null_cells,
        "completeness_pct": completeness,
        "memory_mb": df.memory_usage(deep=True).sum() / (1024 ** 2)
    })

    for col in df.columns:
        nulls = int(df[col].isna().sum())
        n = len(df)
        column_diagnostics.append({
            "dataset": name,
            "column": col,
            "dtype": str(df[col].dtype),
            "non_null": int(df[col].notna().sum()),
            "nulls": nulls,
            "null_pct": (100 * nulls / n) if n else np.nan,
            "unique_non_null": int(df[col].nunique(dropna=True))
        })

overview = pd.DataFrame(overview_rows)
column_quality = pd.DataFrame(column_diagnostics)

display(overview.round({"completeness_pct": 5, "memory_mb": 2}))
display(column_quality.round({"null_pct": 6}))

,dataset,records,columns,distinct_values_sum,full_row_duplicates,null_cells,completeness_pct,memory_mb
0,movies,87585,3,176765,0,0,100.0000,11.9200
1,ratings,32000204,4,26261913,0,0,100.0000,610.3600
2,tags,2000072,4,1499400,0,17,99.9998,145.2100
3,links,87585,3,262595,0,124,99.9528,1.8400


,dataset,column,dtype,non_null,nulls,null_pct,unique_non_null
0,movies,movieId,int32,87585,0,0.0000,87585
1,movies,title,string,87585,0,0.0000,87382
2,movies,genres,string,87585,0,0.0000,1798
3,ratings,userId,int32,32000204,0,0.0000,200948
4,ratings,movieId,int32,32000204,0,0.0000,84432
5,ratings,rating,float32,32000204,0,0.0000,10
6,ratings,timestamp,int64,32000204,0,0.0000,25976523
7,tags,userId,int32,2000072,0,0.0000,15848
8,tags,movieId,int32,2000072,0,0.0000,51323
9,tags,tag,string,2000055,17,0.0008,140979


## Evaluación de llaves candidatas y detección de duplicidad real

Una vez validada la calidad estructural de los datos, el siguiente paso consiste en identificar las posibles llaves de cada tabla y evaluar la existencia de registros duplicados desde una perspectiva relacional.

Esta fase resulta especialmente importante porque el conjunto de datos MovieLens está compuesto por múltiples tablas relacionadas entre sí. Antes de realizar cualquier integración es necesario verificar que los identificadores utilizados como puntos de unión mantengan un comportamiento consistente y que las apariencias de duplicidad correspondan realmente a problemas de calidad y no a características legítimas de la estructura de los datos.

El objetivo principal es distinguir claramente entre:

- Duplicados reales.
- Repeticiones esperadas por el modelo de datos.
- Posibles llaves candidatas.
- Coincidencias textuales que no representan la misma entidad.

## Identificación de llaves candidatas

En un modelo relacional, una llave candidata corresponde a un conjunto mínimo de atributos capaz de identificar de manera única cada observación dentro de una tabla.

La identificación correcta de estas llaves es fundamental porque:

- Permite validar la integridad de los datos.
- Facilita los procesos de integración posteriores.
- Evita uniones incorrectas entre tablas.
- Reduce el riesgo de generar duplicaciones artificiales durante el análisis.

### Tabla movies

La tabla `movies` almacena un registro por película.

Bajo el diseño original de MovieLens, se espera que la variable:

```text
movieId
```

identifique de manera única cada película disponible en el catálogo.

Por esta razón, se evalúa si existen repeticiones del identificador dentro de la tabla.

### Tabla links

La tabla `links` contiene la relación entre cada película y sus identificadores externos.

Al igual que en `movies`, se espera que:

```text
movieId
```

mantenga una relación uno a uno con cada observación.

Por consiguiente, se verifica que no existan múltiples registros asociados al mismo identificador.

### Tabla ratings

La tabla `ratings` representa interacciones entre usuarios y películas.

En este contexto, el identificador de una observación no corresponde a una sola variable, sino a la combinación:

```text
(userId, movieId)
```

Esta pareja describe la calificación emitida por un usuario sobre una película específica.

La evaluación busca determinar si existen combinaciones repetidas que podrían indicar registros duplicados o problemas de consistencia.

### Tabla tags

La situación es diferente para la tabla `tags`.

Un mismo usuario puede:

- Etiquetar una película varias veces.
- Utilizar distintas etiquetas sobre una misma película.
- Registrar etiquetas en momentos diferentes.

Por esta razón no se asume automáticamente una llave primaria compuesta únicamente por `userId` y `movieId`.

En esta etapa se utiliza una referencia más conservadora:

```text
(userId, movieId, tag, timestamp)
```

permitiendo evaluar duplicaciones exactas sin imponer restricciones que la estructura del dataset no garantiza explícitamente.

## Diagnóstico de unicidad

Para cada tabla se generan indicadores que permiten evaluar el comportamiento de las llaves candidatas identificadas.

Entre los principales resultados se obtienen:

### Número total de registros

Corresponde al volumen completo de observaciones contenidas en la tabla.

### Combinaciones distintas de llave

Representa el número de identificadores únicos observados según la llave candidata evaluada.

Este indicador permite determinar si la llave realmente posee capacidad identificadora.

### Filas con llaves repetidas

Cuantifica cuántos registros participan en combinaciones de llave duplicadas.

La existencia de estos casos puede indicar:

- Problemas de calidad de datos.
- Errores de captura.
- Supuestos incorrectos acerca de la granularidad de la tabla.

## Evaluación de títulos repetidos

Una fuente frecuente de errores durante la limpieza de datos consiste en asumir que dos películas con el mismo título representan necesariamente la misma entidad.

Sin embargo, en bases de datos cinematográficas esto rara vez es cierto.

Distintas películas pueden compartir exactamente el mismo nombre debido a factores como:

- Remakes.
- Reestrenos.
- Producciones independientes.
- Adaptaciones realizadas en distintos años.

Por esta razón, la evaluación de títulos repetidos se realiza exclusivamente con fines diagnósticos.

### Detección de coincidencias exactas

Inicialmente se agrupan las películas utilizando el campo:

```text
title
```

tal como aparece en el dataset original.

El objetivo es identificar títulos que aparecen asociados a más de un identificador de película.

Además de contabilizar las repeticiones, se almacenan los correspondientes valores de:

```text
movieId
```

para facilitar su análisis posterior.

## Diagnóstico mediante normalización de títulos

Posteriormente se realiza un segundo ejercicio de comparación utilizando una versión normalizada del título.

La normalización aplicada incluye:

- Eliminación de espacios al inicio y final del texto.
- Conversión a minúsculas mediante comparación insensible a capitalización (`casefold`).

Esta transformación tiene una finalidad exclusivamente diagnóstica y permite detectar diferencias de escritura que podrían ocultar coincidencias aparentes.

Por ejemplo:

```text
Toy Story
```

y

```text
toy story
```

serían tratados como una misma representación textual durante la inspección.

Sin embargo, dicha versión normalizada no sustituye al valor original almacenado en la tabla.

## Principio de conservación de identidad

Un aspecto fundamental de esta etapa consiste en evitar decisiones de limpieza basadas exclusivamente en similitudes textuales.

Por esta razón se adopta la siguiente regla metodológica:

> Dos películas con el mismo título no se consideran duplicadas automáticamente.

La identidad de una película queda definida por su identificador:

```text
movieId
```

y no por el contenido textual del campo `title`.

En consecuencia:

- No se elimina ningún registro por presentar títulos repetidos.
- No se consolidan películas basándose únicamente en coincidencias de nombre.
- Los resultados obtenidos se utilizan únicamente como herramientas de exploración y control de calidad.

## Propósito de esta evaluación

La validación de llaves y la inspección de posibles duplicidades constituyen un paso previo indispensable para los procesos de integración que se realizarán más adelante.

Gracias a este diagnóstico será posible:

- Confirmar la unicidad esperada de los identificadores.
- Comprender la granularidad real de cada tabla.
- Evitar uniones incorrectas durante los procesos de integración.
- Diferenciar entre duplicados reales y repeticiones legítimas.
- Utilizar identificadores confiables como base de las relaciones entre entidades.

Estas verificaciones proporcionan una mayor certeza sobre la estructura relacional del conjunto de datos y establecen los fundamentos necesarios para las etapas posteriores de normalización, enriquecimiento e integración analítica.

In [4]:
def key_report(df, key_columns, table_name):
    duplicate_mask = df.duplicated(subset=key_columns, keep=False)
    return {
        "table": table_name,
        "key": " + ".join(key_columns),
        "rows": len(df),
        "duplicated_key_rows": int(duplicate_mask.sum()),
        "distinct_key_combinations": int(df[key_columns].drop_duplicates().shape[0])
    }

key_checks = pd.DataFrame([
    key_report(movies, ["movieId"], "movies"),
    key_report(links, ["movieId"], "links"),
    key_report(ratings, ["userId", "movieId"], "ratings"),
    key_report(tags, ["userId", "movieId", "tag", "timestamp"], "tags")
])
display(key_checks)

# Diagnóstico de títulos exactos repetidos.
title_counts = (
    movies.groupby("title", dropna=False)
    .agg(record_count=("movieId", "size"),
         movie_ids=("movieId", lambda s: list(s)))
    .query("record_count > 1")
    .sort_values(["record_count", "title"], ascending=[False, True])
    .reset_index()
)

# Normalización temporal para localizar coincidencias de escritura.
movies["title_norm"] = (
    movies["title"].astype("string").str.strip().str.casefold()
)

normalized_title_counts = (
    movies.groupby("title_norm", dropna=False)
    .agg(record_count=("movieId", "size"),
         titles=("title", lambda s: list(s)),
         movie_ids=("movieId", lambda s: list(s)))
    .query("record_count > 1")
    .sort_values("record_count", ascending=False)
    .reset_index()
)

print("Grupos de títulos exactos repetidos:", len(title_counts))
print("Grupos repetidos tras normalización:", len(normalized_title_counts))
display(title_counts.head(30))

,table,key,rows,duplicated_key_rows,distinct_key_combinations
0,movies,movieId,87585,0,87585
1,links,movieId,87585,0,87585
2,ratings,userId + movieId,32000204,0,32000204
3,tags,userId + movieId + tag + timestamp,2000072,0,2000072


Grupos de títulos exactos repetidos: 198
Grupos repetidos tras normalización: 200


,title,record_count,movie_ids
0,Alone (2020),4,"[212511, 223191, 224463, 273117]"
1,Journey to the Center of the Earth (2008),3,"[60514, 192003, 254480]"
2,The Box (2021),3,"[246860, 252868, 276057]"
3,The Plague (2006),3,"[128255, 128261, 206925]"
4,2020 (2020),2,"[230389, 245234]"
5,ATM (2012),2,"[96456, 217057]"
6,Absolution (2015),2,"[136696, 151797]"
7,Adam (2019),2,"[209761, 223916]"
8,Aladdin (1992),2,"[588, 114240]"
9,Alone (2017),2,"[181565, 198719]"


In [19]:
dup = movies[movies.duplicated("title", keep=False)].merge(links, on="movieId", how="left")
grupos = dup.groupby("title").agg(
    n_ids=("movieId", "nunique"),
    n_imdb=("imdbId", "nunique"),
    n_tmdb=("tmdbId", "nunique"),
)
candidatos = grupos[(grupos.n_imdb == 1) | (grupos.n_tmdb == 1)]
len(grupos), len(candidatos)

(198, 20)

In [20]:
grupos[(grupos.n_imdb == 1) | (grupos.n_tmdb == 1)]

,n_ids,n_imdb,n_tmdb
title,,,
Berlin Calling (2008),2,2,1
Clear History (2013),2,2,1
Clockstoppers (2002),2,2,1
Confessions of a Dangerous Mind (2002),2,2,1
Enron: The Smartest Guys in the Room (2005),2,2,1
Eros (2004),2,2,1
God's Angry Man (1980),2,2,1
Good Boys (2019),2,2,1
Lagaan: Once Upon a Time in India (2001),2,2,1


In [23]:
movies.loc[movies["title"].eq("Berlin Calling (2008)")]

,movieId,title,genres
13084,66511,Berlin Calling (2008),Comedy|Drama
56098,191775,Berlin Calling (2008),Comedy|Drama


In [25]:
links[links["movieId"].isin([66511, 191775])]

,movieId,imdbId,tmdbId
13084,66511,1213019,30508
56098,191775,211946,30508


In [32]:
df_aux=ratings[ratings["movieId"].isin([66511, 191775])]
df_aux.groupby("movieId").agg(user_count=("userId", "nunique"))

,user_count
movieId,
66511,98
191775,24


## Cobertura del catálogo y actividad registrada por película

Una vez verificadas las llaves candidatas y la integridad estructural de las tablas principales, el siguiente paso consiste en evaluar la cobertura efectiva del catálogo de películas dentro de los distintos conjuntos de datos que conforman MovieLens.

El objetivo de este análisis es determinar qué proporción de las películas incluidas en el catálogo principal (`movies`) participa realmente en las actividades registradas por los usuarios, ya sea mediante calificaciones (`ratings`) o etiquetas (`tags`).

Esta evaluación permite comprender el nivel de utilización real del catálogo y detectar posibles problemas de integridad referencial antes de iniciar procesos de integración más complejos.

## Importancia de la cobertura del catálogo

La tabla `movies` representa el catálogo maestro del sistema y contiene la lista completa de películas disponibles.

Sin embargo, la mera existencia de una película dentro del catálogo no implica necesariamente que haya generado interacción por parte de los usuarios.

Por esta razón, resulta importante distinguir entre:

- Películas existentes en el catálogo.
- Películas que han recibido al menos una calificación.
- Películas que han recibido al menos una etiqueta.
- Películas asociadas a identificadores externos.

Este análisis permite dimensionar el grado de participación de las películas dentro del ecosistema MovieLens.

## Evaluación mediante identificadores únicos

Para realizar esta comparación se utilizan conjuntos (`set`) de valores únicos de `movieId`.

Este enfoque es particularmente importante porque:

- Una misma película puede recibir miles de calificaciones.
- Una película puede acumular múltiples etiquetas.
- Contar filas de eventos produciría resultados sesgados.

Por ejemplo, si una película recibe 10,000 calificaciones, sigue representando una única película dentro del catálogo.

Por ello, la cobertura se calcula utilizando únicamente identificadores distintos de películas y no el número total de eventos registrados.

## Integridad referencial entre tablas

Además de medir la cobertura, este ejercicio permite validar la consistencia de las relaciones entre tablas.

### Películas presentes en movies pero ausentes en ratings

Una película puede formar parte del catálogo sin haber recibido ninguna calificación.

Este comportamiento es completamente válido y no constituye una anomalía.

Las películas en esta situación simplemente representan elementos disponibles para ser evaluados, pero que aún no registran actividad por parte de los usuarios.

### Películas presentes en movies pero ausentes en tags

De manera similar, una película puede no poseer etiquetas asociadas.

Las etiquetas constituyen información opcional generada voluntariamente por los usuarios, por lo que niveles de cobertura inferiores respecto a las calificaciones son esperables.

### Películas presentes en ratings o tags pero ausentes en movies

Este escenario sí representaría un posible problema de integridad referencial.

Indicaría la existencia de referencias hacia películas no registradas en el catálogo principal, generando inconsistencias en el modelo de datos.

Por esta razón, además de medir la cobertura, se contabilizan los identificadores observados en tablas de actividad que no aparecen en `movies`.

In [5]:
catalog_movie_ids = set(movies["movieId"].dropna().unique())
rating_movie_ids = set(ratings["movieId"].dropna().unique())
tag_movie_ids = set(tags["movieId"].dropna().unique())
link_movie_ids = set(links["movieId"].dropna().unique())

coverage = pd.DataFrame([
    {
        "source": "ratings",
        "distinct_movie_ids": len(rating_movie_ids),
        "catalog_movies_covered": len(catalog_movie_ids & rating_movie_ids),
        "catalog_movies_without_activity": len(catalog_movie_ids - rating_movie_ids),
        "ids_not_in_catalog": len(rating_movie_ids - catalog_movie_ids),
        "catalog_coverage_pct": 100 * len(catalog_movie_ids & rating_movie_ids) / len(catalog_movie_ids)
    },
    {
        "source": "tags",
        "distinct_movie_ids": len(tag_movie_ids),
        "catalog_movies_covered": len(catalog_movie_ids & tag_movie_ids),
        "catalog_movies_without_activity": len(catalog_movie_ids - tag_movie_ids),
        "ids_not_in_catalog": len(tag_movie_ids - catalog_movie_ids),
        "catalog_coverage_pct": 100 * len(catalog_movie_ids & tag_movie_ids) / len(catalog_movie_ids)
    },
    {
        "source": "links",
        "distinct_movie_ids": len(link_movie_ids),
        "catalog_movies_covered": len(catalog_movie_ids & link_movie_ids),
        "catalog_movies_without_activity": len(catalog_movie_ids - link_movie_ids),
        "ids_not_in_catalog": len(link_movie_ids - catalog_movie_ids),
        "catalog_coverage_pct": 100 * len(catalog_movie_ids & link_movie_ids) / len(catalog_movie_ids)
    }
])

display(coverage.round(4))

movies_without_ratings = movies.loc[
    ~movies["movieId"].isin(rating_movie_ids), ["movieId", "title", "genres"]
].copy()

movies_without_tags = movies.loc[
    ~movies["movieId"].isin(tag_movie_ids), ["movieId", "title", "genres"]
].copy()

print("Películas sin calificaciones:", len(movies_without_ratings))
print("Películas sin etiquetas:", len(movies_without_tags))

,source,distinct_movie_ids,catalog_movies_covered,catalog_movies_without_activity,ids_not_in_catalog,catalog_coverage_pct
0,ratings,84432,84432,3153,0,96.4001
1,tags,51323,51323,36262,0,58.5979
2,links,87585,87585,0,0,100.0000


Películas sin calificaciones: 3153
Películas sin etiquetas: 36262


## Cobertura observada del catálogo

Los resultados obtenidos muestran que la integridad referencial del conjunto de datos es consistente.

### Cobertura mediante calificaciones

La tabla `ratings` contiene actividad asociada a:

- **84,432 películas distintas**.
- Lo anterior representa aproximadamente **96.40 %** del catálogo completo.

Esto implica que únicamente:

- **3,153 películas** del catálogo no han recibido ninguna calificación registrada.

Desde una perspectiva analítica, este resultado sugiere una cobertura extremadamente alta del catálogo por parte de los usuarios.

### Cobertura mediante etiquetas

La tabla `tags` presenta una cobertura menor, algo esperado debido al carácter opcional de las etiquetas.

Los resultados muestran:

- **51,323 películas etiquetadas**.
- Una cobertura aproximada de **58.60 %** del catálogo.

En consecuencia:

- **36,262 películas** no presentan etiquetas registradas.

Este comportamiento no debe interpretarse como una deficiencia de calidad, sino como una característica natural de la actividad de etiquetado, la cual requiere una participación más activa por parte de los usuarios.

### Cobertura mediante links

La tabla `links` presenta la cobertura más completa.

Los resultados indican que:

- Las **87,585 películas** del catálogo poseen correspondencia en la tabla de identificadores externos.
- La cobertura alcanza el **100 %**.

Este resultado confirma que todas las películas del catálogo pueden vincularse con sistemas externos como IMDb o TMDb.

## Verificación de consistencia referencial

Uno de los hallazgos más importantes de este análisis es que ninguna de las tres tablas relacionadas presenta identificadores de películas inexistentes en el catálogo principal.

Es decir:

- No existen `movieId` en `ratings` que no estén presentes en `movies`.
- No existen `movieId` en `tags` que no estén presentes en `movies`.
- No existen `movieId` en `links` que no estén presentes en `movies`.

La ausencia de estas discrepancias constituye una evidencia sólida de integridad referencial dentro del conjunto de datos.

## Identificación de películas sin actividad

Finalmente, se generan subconjuntos específicos que contienen las películas que no presentan actividad registrada.

Estas tablas conservan información básica de cada película:

- `movieId`
- `title`
- `genres`

y pueden utilizarse posteriormente para:

- Investigar características de las películas sin interacción.
- Comparar películas activas frente a inactivas.
- Evaluar sesgos de participación dentro de la comunidad de usuarios.
- Analizar la distribución de géneros o periodos de lanzamiento en películas sin actividad.

## Interpretación general

Los resultados obtenidos muestran que el catálogo MovieLens presenta una estructura altamente consistente.

Los principales hallazgos son:

- La cobertura de calificaciones alcanza más del 96 % del catálogo.
- Las etiquetas cubren aproximadamente el 59 % de las películas disponibles.
- La tabla de identificadores externos cubre la totalidad del catálogo.
- No se detectan rupturas de integridad referencial entre las tablas analizadas.
- Las películas sin actividad representan una fracción relativamente pequeña del conjunto total.

En conjunto, estos resultados proporcionan evidencia de que las relaciones entre tablas son consistentes y que el proceso de integración podrá realizarse utilizando `movieId` como llave principal de forma segura y controlada.

## Análisis de valores nulos y evaluación de completitud

Tras validar la estructura general de las tablas y confirmar la consistencia de los identificadores utilizados para la integración, el siguiente paso consiste en analizar detalladamente la presencia de valores faltantes.

Los valores nulos constituyen uno de los aspectos más relevantes dentro de cualquier proceso de calidad de datos. Sin embargo, su mera existencia no implica necesariamente un problema ni justifica de forma automática la eliminación o imputación de registros.

Por este motivo, antes de aplicar cualquier tratamiento, se realiza una evaluación específica orientada a comprender:

- Dónde aparecen los valores faltantes.
- Cuántos registros están afectados.
- Qué significado tendría eliminar o modificar dichas observaciones.
- Cómo impactan sobre la integridad del modelo de datos.

## Identificación sistemática de valores faltantes

Como primer paso, se realiza un recorrido por todas las columnas de las tablas analizadas con el objetivo de identificar aquellas variables que contienen valores nulos.

Para cada caso se calculan dos métricas complementarias:

### Número absoluto de nulos

Representa la cantidad exacta de registros afectados dentro de una variable.

Esta medida permite dimensionar directamente el volumen de información faltante.

### Porcentaje de nulidad

Corresponde a la proporción de registros nulos respecto al tamaño total de la tabla.

A diferencia de la etapa de caracterización general, en esta ocasión el porcentaje se reporta con un mayor nivel de precisión para evitar interpretaciones erróneas.

Esto resulta especialmente importante en conjuntos de datos de gran tamaño, donde cantidades aparentemente pequeñas de registros pueden desaparecer visualmente al redondear los porcentajes.

Por ejemplo, una variable puede contener varios registros nulos y aun así mostrar un porcentaje cercano a cero cuando el tamaño total de la tabla es muy elevado.

In [6]:
null_detail = []
for table_name, df in tables.items():
    for col in df.columns:
        n_null = int(df[col].isna().sum())
        if n_null:
            null_detail.append({
                "table": table_name,
                "column": col,
                "null_count": n_null,
                "null_pct": 100 * n_null / len(df)
            })

null_report = pd.DataFrame(null_detail)
display(null_report.round({"null_pct": 8}))

# Inspección de registros con etiqueta nula.
tags_null = tags.loc[tags["tag"].isna()].copy()
display(tags_null.head(30))

# Inspección de películas sin identificador TMDB.
tmdb_missing = links.loc[links["tmdbId"].isna()].copy()
display(tmdb_missing.head(30))

,table,column,null_count,null_pct
0,tags,tag,17,0.0008
1,links,tmdbId,124,0.1416


,userId,movieId,tag,timestamp
185377,27046,33826,<NA>,1221450908
1394089,89369,281500,<NA>,1670942104
1914668,153443,123,<NA>,1199450867
1914669,153443,346,<NA>,1199451946
1914673,153443,1184,<NA>,1199452261
1914680,153443,1785,<NA>,1199452006
1914681,153443,2194,<NA>,1199450677
1914683,153443,2691,<NA>,1199451002
1914691,153443,4103,<NA>,1199451920
1914693,153443,4473,<NA>,1199451040


,movieId,imdbId,tmdbId
706,721,114103,<NA>
715,730,125877,<NA>
754,770,38426,<NA>
775,791,113610,<NA>
1080,1107,102336,<NA>
1114,1142,116403,<NA>
1282,1316,115548,<NA>
1382,1421,113212,<NA>
1395,1434,123281,<NA>
1571,1630,123953,<NA>


## Resultados identificados

El análisis muestra que únicamente dos variables presentan valores faltantes:

| Tabla | Variable | Registros nulos |
|---------|---------|---------|
| `tags` | `tag` | 17 |
| `links` | `tmdbId` | 124 |

No se detectan valores faltantes en los identificadores principales utilizados para la integración de tablas.

Este resultado es particularmente importante porque garantiza que las relaciones basadas en `movieId` y `userId` pueden construirse sin pérdida de información derivada de claves incompletas.

## Análisis de los registros con etiquetas faltantes

La tabla `tags` contiene únicamente **17 registros** cuyo contenido de etiqueta (`tag`) está ausente.

Considerando el volumen total de observaciones presentes en la tabla, este nivel de nulidad resulta extremadamente reducido.

### Inspección de los registros afectados

Antes de decidir cualquier acción correctiva, se inspeccionan directamente los registros que presentan ausencia de texto en la columna `tag`.

La revisión permite verificar que:

- Los identificadores de usuario permanecen disponibles.
- Los identificadores de película se encuentran completos.
- El momento de creación de la etiqueta sigue registrado mediante `timestamp`.
- La única información ausente corresponde al contenido textual de la etiqueta.

Por tanto, el evento de etiquetado sigue existiendo dentro del sistema, aunque no se disponga del texto asociado.

### Decisión adoptada

No se sustituyen los valores faltantes por cadenas artificiales como:

```text
"unknown"
```

o

```text
"sin_etiqueta"
```

ya que dichas categorías no forman parte de la información original distribuida por MovieLens.

Asimismo, los registros no se eliminan de la tabla base.

Su tratamiento quedará restringido a análisis específicos que requieran necesariamente el contenido textual de las etiquetas.

## Análisis de identificadores TMDB faltantes

La segunda fuente de nulidad identificada corresponde a la variable:

```text
tmdbId
```

dentro de la tabla `links`.

Se detectan **124 películas** sin correspondencia en The Movie Database (TMDb).

### Evaluación del impacto

La ausencia de un identificador TMDb no implica la pérdida de la película dentro del ecosistema MovieLens.

Cada uno de estos registros mantiene disponible:

- `movieId`
- `imdbId`

Por tanto, la película continúa siendo identificable tanto dentro del catálogo interno como mediante referencias externas alternativas.

### Interpretación

Existen múltiples razones legítimas por las cuales una película podría carecer de identificador TMDb:

- Ausencia temporal en dicha plataforma.
- Cambios en la catalogación histórica.
- Diferencias entre sistemas de referencia.
- Problemas de sincronización entre catálogos externos.

En cualquier caso, la falta de un `tmdbId` no compromete la integridad del modelo principal de datos.

### Decisión adoptada

Los registros afectados se conservan íntegramente.

La ausencia de un identificador externo no justifica la eliminación de películas del catálogo, especialmente cuando los identificadores principales continúan disponibles.

## Principio de conservación de datos

Siguiendo los criterios metodológicos definidos para este proyecto, se adopta una estrategia de preservación de la información original.

En consecuencia:

- No se eliminan registros debido a valores nulos.
- No se imputan categorías artificiales.
- No se reemplazan identificadores faltantes.
- No se modifica el contenido original de las tablas base.

Este enfoque permite mantener la trazabilidad completa de los datos distribuidos por MovieLens.

## Implicaciones para análisis posteriores

Aunque los registros se conservan, es importante reconocer que ciertas actividades analíticas requerirán tratamientos específicos.

Por ejemplo:

- Los análisis de texto deberán excluir explícitamente registros donde `tag` sea nulo.
- Las integraciones con TMDb deberán limitarse a películas que dispongan de un `tmdbId` válido.
- Los procesos de enriquecimiento externo deberán documentar claramente las observaciones descartadas.

Estas exclusiones se realizarán únicamente en el contexto de cada análisis particular y nunca modificando las tablas base del proyecto.

## Conclusiones

La evaluación de completitud indica que el conjunto MovieLens 32M presenta niveles muy altos de calidad estructural.

Los principales hallazgos son:

- La cantidad de valores nulos es extremadamente reducida respecto al volumen total de datos.
- No existen valores faltantes en los identificadores principales de integración.
- Los nulos detectados corresponden a variables secundarias y no comprometen la integridad relacional del modelo.
- La conservación de los registros permite mantener la trazabilidad completa de la información original.
- Cualquier tratamiento posterior sobre valores faltantes se realizará de forma contextual y documentada.

En consecuencia, no se aplicarán transformaciones destructivas sobre las tablas base y se continuará con las siguientes etapas de limpieza e integración preservando íntegramente la información original distribuida por MovieLens.

## Correspondencia de identificadores y análisis de actividad de usuarios

Después de verificar la calidad estructural de los datos y evaluar la completitud de los atributos, el siguiente paso consiste en validar la consistencia de los identificadores utilizados para relacionar las distintas tablas del ecosistema MovieLens.

Esta etapa tiene dos objetivos principales:

1. Comprobar que las relaciones entre películas mantienen integridad referencial.
2. Comprender cómo participan los usuarios en las distintas actividades registradas por la plataforma.

La correcta interpretación de estos identificadores resulta esencial para garantizar que las integraciones posteriores se realicen de forma segura y sin pérdida de información.

## Correspondencia de identificadores de películas

La variable `movieId` constituye el identificador interno principal del conjunto de datos MovieLens.

Este atributo aparece en las cuatro tablas analizadas:

- `movies`
- `ratings`
- `tags`
- `links`

Por lo tanto, representa la pieza central del modelo relacional y será la clave utilizada para la integración entre entidades a lo largo del proyecto.

### Validación de integridad referencial

Con el fin de verificar la consistencia del modelo, se comparan los conjuntos de identificadores presentes en:

- La tabla de catálogo (`movies`).
- Las tablas de actividad (`ratings` y `tags`).
- La tabla de identificadores externos (`links`).

Este breve análisis busca responder una pregunta fundamental:

> ¿Todos los identificadores de películas observados en las tablas dependientes existen realmente dentro del catálogo oficial de películas?

Para ello se calculan:

- Número de películas distintas observadas en cada tabla.
- Número de identificadores encontrados en el catálogo.
- Número de identificadores huérfanos o sin correspondencia.

In [7]:
# Integridad referencial de movieId.
referential_report = pd.DataFrame([
    {
        "child_table": "ratings",
        "distinct_movie_ids": len(rating_movie_ids),
        "ids_found_in_movies": len(rating_movie_ids & catalog_movie_ids),
        "orphan_movie_ids": len(rating_movie_ids - catalog_movie_ids)
    },
    {
        "child_table": "tags",
        "distinct_movie_ids": len(tag_movie_ids),
        "ids_found_in_movies": len(tag_movie_ids & catalog_movie_ids),
        "orphan_movie_ids": len(tag_movie_ids - catalog_movie_ids)
    },
    {
        "child_table": "links",
        "distinct_movie_ids": len(link_movie_ids),
        "ids_found_in_movies": len(link_movie_ids & catalog_movie_ids),
        "orphan_movie_ids": len(link_movie_ids - catalog_movie_ids)
    }
])
display(referential_report)

rating_users = set(ratings["userId"].unique())
tag_users = set(tags["userId"].unique())

user_activity = pd.DataFrame([{
    "users_with_ratings": len(rating_users),
    "users_with_tags": len(tag_users),
    "users_in_both": len(rating_users & tag_users),
    "users_only_ratings": len(rating_users - tag_users),
    "users_only_tags": len(tag_users - rating_users),
    "union_users": len(rating_users | tag_users),
    "taggers_pct_of_raters": 100 * len(rating_users & tag_users) / len(rating_users)
}])
display(user_activity)

,child_table,distinct_movie_ids,ids_found_in_movies,orphan_movie_ids
0,ratings,84432,84432,0
1,tags,51323,51323,0
2,links,87585,87585,0


,users_with_ratings,users_with_tags,users_in_both,users_only_ratings,users_only_tags,union_users,taggers_pct_of_raters
0,200948,15848,15848,185100,0,200948,7.8866


### Resultados obtenidos

Los resultados muestran un comportamiento completamente consistente.

#### Tabla ratings

- Se identifican **84,432 películas distintas** con al menos una calificación.
- Todas las películas calificadas aparecen registradas en el catálogo.
- No se detectan identificadores huérfanos.

#### Tabla tags

- Se observan **51,323 películas distintas** asociadas a etiquetas.
- Todas ellas poseen correspondencia en la tabla `movies`.
- No existen referencias inválidas.

#### Tabla links

- Las **87,585 películas** del catálogo poseen correspondencia dentro de la tabla de identificadores externos.
- La cobertura es completa.
- No se observan inconsistencias referenciales.

### Interpretación

La ausencia de identificadores huérfanos constituye una evidencia sólida de integridad referencial.

Esto significa que:

- Toda película calificada existe en el catálogo.
- Toda película etiquetada existe en el catálogo.
- Toda película asociada a identificadores externos existe en el catálogo.

Desde la perspectiva de integración de datos, este resultado indica que `movieId` puede utilizarse con confianza como llave principal para relacionar información entre tablas.

## Consideraciones sobre identificadores externos

Además de `movieId`, el dataset incorpora identificadores externos provenientes de otras plataformas:

### IMDb

Representado mediante:

```text
imdbId
```

### The Movie Database (TMDb)

Representado mediante:

```text
tmdbId
```

Estos identificadores permiten enriquecer la información utilizando fuentes externas y facilitan futuras integraciones con otros repositorios cinematográficos.

No obstante, es importante señalar que dichos atributos no sustituyen la identidad principal definida por MovieLens.

En consecuencia:

- `movieId` continúa siendo la llave principal del proyecto.
- `imdbId` y `tmdbId` se consideran atributos complementarios.
- Cualquier integración basada exclusivamente en identificadores externos requerirá validaciones adicionales.

## Correspondencia de identificadores de usuarios

La segunda parte del análisis se centra en los usuarios registrados dentro de las tablas de actividad.

A diferencia de las películas, los usuarios únicamente aparecen en:

- `ratings`
- `tags`

Las tablas `movies` y `links` describen entidades relacionadas con películas y no contienen información sobre usuarios.

Por este motivo, resulta relevante estudiar cómo se distribuye la participación de los usuarios entre ambas actividades.

## Comparación de usuarios por tipo de actividad

Se construyen conjuntos únicos de `userId` para responder las siguientes preguntas:

- ¿Cuántos usuarios emiten calificaciones?
- ¿Cuántos usuarios generan etiquetas?
- ¿Cuántos realizan ambas actividades?
- ¿Existen usuarios que únicamente etiqueten y nunca califiquen?

### Usuarios que califican películas

La tabla `ratings` contiene actividad de:

**200,948 usuarios distintos.**

Este grupo representa la base completa de usuarios observados dentro del conjunto de datos.

### Usuarios que utilizan etiquetas

La tabla `tags` registra actividad de:

**15,848 usuarios distintos.**

Este resultado muestra que la actividad de etiquetado es considerablemente menos frecuente que la emisión de calificaciones.

### Usuarios que realizan ambas actividades

Todos los usuarios que generan etiquetas aparecen también en la tabla de calificaciones.

Se identifican:

**15,848 usuarios que califican y etiquetan películas.**

### Usuarios exclusivos de calificaciones

Adicionalmente se observa que:

**185,100 usuarios califican películas sin registrar etiquetas.**

Este grupo constituye la gran mayoría de la comunidad de usuarios.

### Usuarios exclusivos de etiquetas

No se detectan usuarios que etiqueten películas sin haber emitido previamente alguna calificación.

El número de usuarios exclusivos de etiquetado es:

**0 usuarios.**

## Interpretación del comportamiento observado

Los resultados sugieren una estructura coherente de participación dentro de la plataforma MovieLens.

La evidencia indica que:

- La calificación de películas es la actividad predominante.
- El etiquetado representa una acción más especializada y menos frecuente.
- Todos los usuarios que etiquetan también participan mediante calificaciones.
- Aproximadamente el **7.89 %** de los usuarios que califican películas generan además etiquetas.

Este comportamiento es consistente con la naturaleza de ambas acciones.

Emitir una calificación requiere un esfuerzo mínimo por parte del usuario, mientras que asignar etiquetas implica una participación más activa y una mayor inversión de tiempo.

Por tanto, la menor cantidad de usuarios presentes en la tabla `tags` no constituye una anomalía ni un problema de calidad de datos.

## Conclusiones

La evaluación realizada permite confirmar varios aspectos fundamentales para las siguientes etapas de integración:

- `movieId` mantiene integridad referencial completa entre las tablas analizadas.
- No existen identificadores de películas huérfanos.
- Los identificadores externos funcionan como atributos complementarios y no sustituyen la llave principal del modelo.
- La actividad de etiquetado representa un subconjunto especializado de los usuarios que califican películas.
- No se detectan inconsistencias entre los conjuntos de usuarios observados en las tablas de actividad.

En conjunto, estos resultados proporcionan evidencia adicional de que el modelo relacional distribuido por MovieLens presenta una estructura sólida y consistente para avanzar hacia las etapas de normalización, enriquecimiento e integración analítica.

## Validación de la escala de calificaciones

Una vez verificada la integridad estructural de las tablas y la consistencia de los identificadores utilizados en el modelo relacional, se procede a validar uno de los atributos más importantes del conjunto de datos: la variable `rating`.

Las calificaciones constituyen la principal fuente de información sobre las preferencias de los usuarios y serán utilizadas en múltiples análisis posteriores.

Por esta razón, antes de realizar cualquier agregación o integración analítica, resulta indispensable confirmar que todas las observaciones cumplen las reglas definidas por la documentación oficial de MovieLens.

## Escala documentada por MovieLens

Según la documentación del conjunto de datos, las calificaciones deben pertenecer a una escala discreta de diez niveles comprendida entre **0.5** y **5.0**, utilizando incrementos de medio punto.

La escala permitida es la siguiente:

```text
0.5
1.0
1.5
2.0
2.5
3.0
3.5
4.0
4.5
5.0
```

En consecuencia, cualquier valor fuera de este conjunto representaría una inconsistencia que debería ser investigada antes de continuar con el análisis.

## Verificación de valores observados

La primera validación consiste en identificar todos los valores distintos presentes en la columna `rating` y compararlos con los valores permitidos por la especificación oficial.

Esta comprobación permite responder preguntas fundamentales como:

- ¿Se encuentran presentes todos los niveles de la escala?
- ¿Existen valores inesperados?
- ¿Se produjeron errores durante la carga de los datos?
- ¿Hay observaciones que contradigan la documentación oficial?

Los resultados deberían mostrar que los valores observados coinciden exactamente con los diez niveles permitidos por MovieLens.

## Detección de registros fuera de escala

Posteriormente se realiza una validación registro por registro para identificar posibles observaciones cuyo valor de calificación no pertenezca al conjunto permitido.

Esta comprobación resulta más robusta que simplemente contar valores únicos, ya que garantiza que cada observación individual cumple las reglas definidas por el sistema de calificación.

In [8]:
allowed_ratings = np.arange(0.5, 5.0 + 0.5, 0.5)
observed_ratings = np.sort(ratings["rating"].dropna().unique())
invalid_rating_mask = ~ratings["rating"].isin(allowed_ratings)

rating_validation = {
    "allowed_values": allowed_ratings.tolist(),
    "observed_values": observed_ratings.tolist(),
    "invalid_rows": int(invalid_rating_mask.sum()),
    "minimum": float(ratings["rating"].min()),
    "maximum": float(ratings["rating"].max())
}
display(pd.Series(rating_validation, name="value"))

rating_distribution = (
    ratings["rating"].value_counts(dropna=False)
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="frequency")
)
rating_distribution["percentage"] = (
    100 * rating_distribution["frequency"] / len(ratings)
)
display(rating_distribution)

if invalid_rating_mask.any():
    invalid_ratings = ratings.loc[invalid_rating_mask].copy()
    invalid_ratings.to_csv(
        DIAGNOSTICS_DIR / "ratings_fuera_de_escala.csv",
        index=False, encoding="utf-8"
    )
    warnings.warn("Se detectaron calificaciones fuera de la escala documentada.")
else:
    print("Validación satisfactoria: todas las calificaciones pertenecen a la escala.")

allowed_values     [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, ...
observed_values    [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, ...
invalid_rows                                                       0
minimum                                                       0.5000
maximum                                                       5.0000
Name: value, dtype: object

,rating,frequency,percentage
0,0.5000,525132,1.6410
1,1.0000,946675,2.9583
2,1.5000,531063,1.6596
3,2.0000,2028622,6.3394
4,2.5000,1685386,5.2668
5,3.0000,6054990,18.9217
6,3.5000,4290105,13.4065
7,4.0000,8367654,26.1488
8,4.5000,2974000,9.2937
9,5.0000,4596577,14.3642


Validación satisfactoria: todas las calificaciones pertenecen a la escala.


## Resultado obtenido

La evaluación identifica:

```text
Registros fuera de escala: 0
```

Esto significa que ninguna calificación registrada infringe las restricciones documentadas por MovieLens.

Por tanto:

- No existen valores inválidos.
- No se detectan errores de captura evidentes.
- No hay indicios de corrupción en la variable `rating`.

### Validación de límites mínimos y máximos

Como comprobación adicional se analizan los valores extremos observados dentro de la distribución.

Los resultados muestran:

| Indicador | Valor |
|------------|------------|
| Calificación mínima | 0.5 |
| Calificación máxima | 5.0 |

Estos límites coinciden exactamente con la escala oficial del dataset.

La coincidencia entre los valores extremos observados y los límites documentados refuerza la evidencia de que la variable fue registrada correctamente.

### Distribución de frecuencias de las calificaciones

Además de validar la escala, se analiza la frecuencia con la que aparece cada nivel de calificación dentro del conjunto de datos.

Esta evaluación proporciona una primera aproximación al comportamiento general de los usuarios y permite detectar posibles anomalías en la distribución.

### Principales hallazgos

Los resultados muestran que las puntuaciones positivas dominan claramente la distribución.

Las categorías más frecuentes corresponden a:

| Calificación | Frecuencia aproximada |
|--------------|--------------|
| 4.0 | 8.37 millones |
| 3.0 | 6.05 millones |
| 5.0 | 4.60 millones |
| 3.5 | 4.29 millones |

Por el contrario, las puntuaciones más bajas presentan frecuencias considerablemente menores.

Por ejemplo:

- Las calificaciones de 0.5 representan aproximadamente el 1.64 % del total.
- Las calificaciones de 1.0 representan alrededor del 2.96 %.

### Interpretación preliminar

La distribución observada sugiere una tendencia positiva en las evaluaciones realizadas por los usuarios.

Este comportamiento es frecuente en sistemas de recomendación, donde los usuarios tienden a interactuar principalmente con contenidos que les resultan interesantes o satisfactorios.

No obstante, en esta etapa el análisis tiene únicamente un propósito de validación y descripción de la variable. La interpretación profunda de los patrones de valoración será desarrollada en las fases posteriores de análisis exploratorio.

## Estrategia ante posibles valores inválidos

Aunque no se detectaron registros fuera de escala, se establece explícitamente un protocolo de actuación para futuras revisiones de calidad.

Si se hubieran identificado observaciones inválidas:

- No se corregirían automáticamente.
- No se redondearían las calificaciones.
- No se modificarían los datos originales.

En su lugar:

1. Los registros serían aislados en un archivo específico de auditoría.
2. Se documentaría el incidente.
3. Se conservarían los datos originales para mantener la trazabilidad.
4. La decisión de tratamiento se tomaría posteriormente con evidencia suficiente.

Este enfoque evita introducir modificaciones arbitrarias en información cuyo origen o significado podría requerir una investigación adicional.

## Conclusiones

La validación realizada confirma que la variable `rating` cumple completamente las especificaciones documentadas por MovieLens.

Los principales resultados obtenidos son:

- Los valores observados coinciden exactamente con la escala oficial.
- No existen registros fuera de rango.
- Los límites mínimo y máximo corresponden a los definidos por la documentación.
- Se encuentran representados los diez niveles de calificación permitidos.
- No fue necesario aplicar correcciones ni transformaciones sobre la variable original.

En consecuencia, la columna `rating` puede considerarse una variable de alta calidad y apta para ser utilizada en los análisis descriptivos, estadísticos y de integración que se desarrollarán en las siguientes etapas del proyecto.

## Normalización de la variable `genres` y construcción de la tabla puente película–género

Una vez verificadas la calidad, completitud e integridad de las tablas principales, el siguiente paso consiste en abordar uno de los aspectos más importantes de la estructura del catálogo de películas: la representación de los géneros cinematográficos.

Esta etapa tiene como objetivo transformar una estructura desnormalizada en una representación relacional más adecuada para análisis posteriores, preservando en todo momento la información original distribuida por MovieLens.

La normalización de géneros permitirá realizar consultas, agregaciones y análisis comparativos de forma mucho más eficiente y consistente.

## Problema de modelado de la variable `genres`

En la tabla `movies`, los géneros de una película no se almacenan en registros independientes, sino dentro de una única celda de texto.

Por ejemplo:

```text
Action|Adventure|Sci-Fi
```

Desde una perspectiva descriptiva, este formato resulta compacto y fácilmente interpretable por una persona.

Sin embargo, desde el punto de vista analítico presenta diversas limitaciones.

### Limitaciones de la representación original

Cuando varios géneros se almacenan dentro de una misma celda:

- No es posible contar géneros directamente.
- Resulta difícil calcular estadísticas por género.
- Las agrupaciones requieren procesamiento adicional.
- Se complica la comparación entre categorías.
- Las integraciones con otras tablas requieren transformaciones previas.

Por ejemplo, responder preguntas como:

- ¿Cuántas películas pertenecen al género Acción?
- ¿Cuál es la valoración promedio de las películas de Ciencia Ficción?
- ¿Qué género recibe más etiquetas?
- ¿Qué género concentra más interacciones de usuarios?

requiere primero separar los géneros individuales contenidos en cada registro.

## Relación muchos a muchos entre películas y géneros

Conceptualmente, la relación existente entre películas y géneros corresponde a una relación de tipo muchos a muchos.

Esto significa que:

### Una película puede pertenecer a varios géneros

Por ejemplo:

```text
Toy Story
```

puede clasificarse simultáneamente como:

- Animation
- Adventure
- Comedy
- Children
- Fantasy

### Un género puede agrupar muchas películas

Por ejemplo:

```text
Drama
```

aparece asociado a miles de películas diferentes dentro del catálogo.

La estructura original de `movies` almacena esta relación comprimida en una única columna, dificultando su explotación analítica.

## Construcción de una tabla puente

Para representar adecuadamente esta relación se genera una tabla puente denominada:

```text
movie_genres
```

El objetivo es producir una estructura normalizada donde cada fila represente una única relación entre una película y un género.

La nueva estructura adopta la siguiente forma:

| movieId | genre |
|----------|----------|
| 1 | Adventure |
| 1 | Animation |
| 1 | Children |
| 1 | Comedy |
| 1 | Fantasy |

De esta manera, una película con cinco géneros generará cinco registros independientes dentro de la tabla puente.

## Conservación de la información original

La normalización no implica la eliminación de la información existente.

Antes de realizar cualquier transformación se conserva una copia de la columna original mediante:

```text
genres_original
```

Esta decisión responde a los principios metodológicos definidos para el proyecto:

- Preservar la información original.
- Mantener trazabilidad completa de las transformaciones.
- Permitir auditorías futuras.
- Facilitar la reproducción exacta del proceso de limpieza.

La columna original continúa disponible para referencias posteriores y para verificar cualquier transformación realizada.

## Proceso de expansión de géneros

La normalización se desarrolla mediante varias etapas:

### Separación de categorías

Cada cadena de texto se divide utilizando el carácter:

```text
|
```

que actúa como separador entre géneros.

### Expansión de registros

Posteriormente, cada género individual se transforma en una fila independiente mediante una operación de explosión de listas (`explode`).

### Limpieza básica

Se eliminan:

- Espacios residuales.
- Cadenas vacías.
- Posibles duplicados exactos de la combinación `(movieId, genre)`.

Como resultado se obtiene una representación limpia y normalizada de la relación película–género.

### Tratamiento de `(no genres listed)`

En la columna `genres`, MovieLens usa la etiqueta especial `(no genres listed)` para las películas a las que no se les asignó ningún género. Aunque aparece como texto, en la práctica representa un **valor faltante**.

Este faltante pasa inadvertido porque no está codificado como nulo: `isna()` no lo detecta, y si no se trata, el análisis lo consideraría un género más. Por eso, antes de continuar, cuantificamos cuántas películas y calificaciones involucra, analizamos su impacto sobre los resultados y, con base en eso, decidimos cómo tratarlas.

In [9]:
SIN_GENERO = "(no genres listed)"

mask_sin_genero = movies["genres"].eq(SIN_GENERO)
print(mask_sin_genero.sum(), round(mask_sin_genero.mean() * 100, 2), "% de las películas")


7080 8.08 % de las películas


Del total de películas del catálogo, **7,080 (8.08%)** tienen la etiqueta `(no genres listed)`, es decir, no cuentan con ningún género asignado.

### Distribucion de los géneros considerando `(no genres listed)`

In [10]:


movie_genres = (
    movies[["movieId", "genres"]]
    .assign(genre=lambda d: d["genres"].fillna("").str.split("|"))
    .explode("genre")
    [["movieId", "genre"]]
    .assign(genre=lambda d: d["genre"].astype("string").str.strip())
)

movie_genres = movie_genres.loc[
    movie_genres["genre"].notna() & movie_genres["genre"].ne("")
].drop_duplicates(["movieId", "genre"]).reset_index(drop=True)

genre_frequency = (
    movie_genres.groupby("genre")
    .agg(movie_count=("movieId", "nunique"))
    .sort_values("movie_count", ascending=False)
    .reset_index()
)
genre_frequency["catalog_pct"] = (
    100 * genre_frequency["movie_count"] / movies["movieId"].nunique()
)

print("Número de géneros distintos:", movie_genres["genre"].nunique())
print("Filas de la tabla puente:", len(movie_genres))
display(genre_frequency)


Número de géneros distintos: 20
Filas de la tabla puente: 154170


,genre,movie_count,catalog_pct
0,Drama,34175,39.0192
1,Comedy,23124,26.4018
2,Thriller,11823,13.4989
3,Romance,10369,11.8388
4,Action,9668,11.0384
5,Documentary,9363,10.6902
6,Horror,8654,9.8807
7,(no genres listed),7080,8.0836
8,Crime,6976,7.9648
9,Adventure,5402,6.1677


### Distribución de los géneros (incluyendo `(no genres listed)`)

Drama es el género más frecuente del catálogo: aparece en **34,175 películas (39.0%)**, seguido de Comedy con **23,124 (26.4%)** y Thriller con **11,823 (13.5%)**. Los siguientes géneros están muy cerca entre sí, con proporciones de entre 9.9% y 11.8% (Romance, Action, Documentary y Horror).

La categoría `(no genres listed)` ocupa el octavo lugar, con **7,080 películas (8.1%)**, una frecuencia comparable a la de géneros reales como Horror.

Los porcentajes **no suman 100%** porque una misma película puede tener varios géneros y se cuenta en cada uno de ellos. Estas cifras describen la composición del catálogo, no la cantidad de calificaciones por género.

### Imputación 
Se opta por un enfoque conservador, se imputara con la moda 

In [11]:
SIN_GENERO = "(no genres listed)"
mask_sin_genero = movies["genres"].eq(SIN_GENERO)
genero_moda = (
    movies.loc[~mask_sin_genero, "genres"]
    .str.split("|").explode().str.strip()
    .value_counts().idxmax()
)
print("Moda:", genero_moda)
movies_imp = movies.copy()
movies_imp["genre_imputed"] = mask_sin_genero #se pone una etiqueta sobre el registro para indicar si es faltante o no
movies_imp.loc[mask_sin_genero, "genres"] = genero_moda

Moda: Drama


In [51]:
movies_imp.head()

,movieId,title,genres,genres_original,genre_imputed
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure|Animation|Children|Comedy|Fantasy,False
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure|Children|Fantasy,False
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy|Romance,False
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy|Drama|Romance,False
4,5,Father of the Bride Part II (1995),Comedy,Comedy,False


In [12]:
movie_genres_imp = (
    movies_imp[["movieId", "genres"]]
    .assign(genre=lambda d: d["genres"].fillna("").str.split("|"))
    .explode("genre")
    [["movieId", "genre"]]
    .assign(genre=lambda d: d["genre"].astype("string").str.strip())
)

genre_frequency_imp = (
    movie_genres_imp.groupby("genre")
    .agg(movie_count=("movieId", "nunique"))
    .sort_values("movie_count", ascending=False)
    .reset_index()
)
genre_frequency_imp["catalog_pct"] = (
    100 * genre_frequency_imp["movie_count"] / movies_imp["movieId"].nunique()
)

display(genre_frequency_imp)

,genre,movie_count,catalog_pct
0,Drama,41255,47.1028
1,Comedy,23124,26.4018
2,Thriller,11823,13.4989
3,Romance,10369,11.8388
4,Action,9668,11.0384
5,Documentary,9363,10.6902
6,Horror,8654,9.8807
7,Crime,6976,7.9648
8,Adventure,5402,6.1677
9,Sci-Fi,4907,5.6026


### Distribución de los géneros después de la imputación

Al imputar con la moda las 7,080 películas sin género, **todas** quedaron asignadas a Drama, que era el género más frecuente. Como consecuencia, Drama pasó de **34,175 películas (39.0%)** a **41,255 (47.1%)**, un aumento de 8.1 puntos porcentuales (cerca de 21% en términos relativos). Los demás géneros no cambian: Comedy se mantiene en 26.4% y Thriller en 13.5%.

El **orden de los géneros se conserva** (Drama, Comedy, Thriller, Romance, Action...), pero la brecha entre Drama y Comedy se ensancha de unos 12.6 a unos 20.7 puntos. La imputación, por tanto, no altera el ranking, pero sí **infla la presencia de Drama** en el catálogo 


## Beneficios analíticos de la normalización

La construcción de la tabla puente proporciona una estructura significativamente más adecuada para el análisis exploratorio y descriptivo.

Gracias a esta normalización será posible:

- Contar películas por género de forma directa.
- Calcular calificaciones promedio por categoría.
- Analizar actividad de usuarios por género.
- Estudiar distribución de etiquetas según categorías cinematográficas.
- Construir visualizaciones comparativas.
- Integrar géneros con métricas agregadas de películas.

Todo ello sin alterar ni perder la representación original almacenada en la tabla `movies`.

## Conclusiones

La normalización de la variable `genres` constituye una de las transformaciones estructurales más importantes del proyecto.

Los principales resultados obtenidos son:

- Se preserva la columna original de géneros.
- Se construye una tabla puente normalizada película–género.
- Se identifican 20 géneros distintos en el catálogo.
- Se generan 154,170 relaciones película–género.
- Se habilita una estructura adecuada para futuras agregaciones e integraciones analíticas.


## Construcción de métricas agregadas por película

Una vez verificadas las relaciones entre tablas, evaluada la cobertura del catálogo y normalizada la información de géneros, el siguiente paso consiste en generar métricas agregadas a nivel de película.

El objetivo de esta etapa es enriquecer la información disponible para cada película sin alterar la granularidad original del catálogo.

Mientras que las tablas `ratings` y `tags` contienen millones de eventos individuales generados por los usuarios, la tabla `movies` mantiene una estructura mucho más estable, con una sola fila por película. Por esta razón, antes de realizar cualquier integración, es necesario resumir la actividad registrada en dichas tablas hacia el nivel de análisis deseado.

## Motivación de los agregados

El conjunto de datos MovieLens fue diseñado siguiendo una estructura transaccional.

En este modelo:

- Cada fila de `ratings` representa una calificación individual.
- Cada fila de `tags` representa una aplicación de etiqueta realizada por un usuario.

Como consecuencia, una misma película puede encontrarse representada miles o incluso millones de veces dentro de las tablas de eventos.

Intentar unir directamente estas tablas con el catálogo de películas provocaría problemas importantes:

- Multiplicación artificial de filas.
- Crecimiento innecesario del volumen de datos.
- Aparición de relaciones many-to-many accidentales.
- Distorsión de estadísticas posteriores.

Por este motivo, antes de la integración se construyen tablas resumen que condensan la actividad observada para cada película.

## Agregación de calificaciones

La tabla `ratings` constituye la principal fuente de información sobre la valoración que los usuarios otorgan a las películas.

A partir de ella se generan múltiples indicadores descriptivos agrupados por:

```text
movieId
```

De este modo, cada película quedará representada por una única fila dentro de la tabla agregada.

### Número de calificaciones

```text
rating_count
```

Corresponde al número total de valoraciones recibidas por una película.

Este indicador puede interpretarse como una medida de popularidad o nivel de participación de los usuarios.

### Calificación promedio

```text
rating_mean
```

Representa la valoración media otorgada por los usuarios.

Este será uno de los indicadores más importantes para futuros análisis de calidad percibida y preferencias del público.

### Desviación estándar

```text
rating_std
```

Mide la dispersión de las calificaciones.

Valores elevados indican desacuerdo entre los usuarios, mientras que valores bajos sugieren opiniones más homogéneas.

### Calificación mínima

```text
rating_min
```

Permite identificar la valoración más baja registrada para una película.

### Calificación máxima

```text
rating_max
```

Permite conocer la valoración más alta recibida.

### Número de usuarios calificadores

```text
rating_users
```

Corresponde al número de usuarios distintos que emitieron al menos una calificación para la película.

Esta métrica complementa la información de popularidad al eliminar posibles efectos producidos por registros repetidos del mismo usuario.

## Agregación de etiquetas

La tabla `tags` contiene información semántica generada por los usuarios mediante etiquetas descriptivas.

Al igual que en el caso de las calificaciones, se construye un resumen agrupado por `movieId`.

### Número de aplicaciones de etiquetas

```text
tag_application_count
```

Representa el número total de eventos de etiquetado registrados para cada película.

Cada aplicación de una etiqueta se contabiliza como una observación independiente.

### Número de etiquetas no nulas

```text
non_null_tag_count
```

Cuenta únicamente aquellas aplicaciones donde existe un valor textual válido para la etiqueta.

Esta separación resulta relevante debido a la presencia de algunos registros con valores nulos identificados durante la evaluación de calidad.

### Número de etiquetas distintas

```text
distinct_tags
```

Indica la diversidad semántica de las etiquetas asociadas a una película.

Valores elevados sugieren que la comunidad utiliza una amplia variedad de términos para describir el contenido o características de la obra.

### Número de usuarios etiquetadores

```text
tag_users
```

Representa el número de usuarios distintos que han interactuado mediante etiquetas con una determinada película.

Este indicador proporciona una visión complementaria de la participación de la comunidad.

## Preservación de la granularidad

Un principio fundamental de esta etapa consiste en evitar modificaciones accidentales de la granularidad de los datos.

Por esta razón, tanto `rating_agg` como `tag_agg` deben cumplir una condición crítica:

> Cada tabla agregada debe contener como máximo una fila por película.

La validación posterior confirma que este requisito se cumple correctamente.

In [13]:
rating_agg = (
    ratings.groupby("movieId")
    .agg(
        rating_count=("rating", "size"),
        rating_mean=("rating", "mean"),
        rating_std=("rating", "std"),
        rating_min=("rating", "min"),
        rating_max=("rating", "max"),
        rating_users=("userId", "nunique")
    )
    .reset_index()
)

tag_agg = (
    tags.groupby("movieId")
    .agg(
        tag_application_count=("tag", "size"),
        non_null_tag_count=("tag", "count"),
        distinct_tags=("tag", "nunique"),
        tag_users=("userId", "nunique")
    )
    .reset_index()
)

print("Filas en agregado de ratings:", len(rating_agg))
print("Filas en agregado de tags:", len(tag_agg))
print("¿rating_agg tiene movieId único?:", rating_agg["movieId"].is_unique)
print("¿tag_agg tiene movieId único?:", tag_agg["movieId"].is_unique)

Filas en agregado de ratings: 84432
Filas en agregado de tags: 51323
¿rating_agg tiene movieId único?: True
¿tag_agg tiene movieId único?: True


### Resultado de la validación

La tabla de agregados de calificaciones contiene:

```text
84,432 películas
```

y presenta unicidad completa de `movieId`.

Asimismo, la tabla de agregados de etiquetas contiene:

```text
51,323 películas
```

también con unicidad garantizada para `movieId`.

Los resultados muestran:

```text
rating_agg["movieId"].is_unique = True
tag_agg["movieId"].is_unique = True
```

lo que confirma que ambas estructuras son adecuadas para futuras integraciones uno a uno con el catálogo.

## Interpretación de valores faltantes en métricas agregadas

Un aspecto importante de estas métricas es la correcta interpretación de la ausencia de actividad.

Por ejemplo, una película que nunca ha recibido calificaciones presentará:

```text
rating_count = 0
```

Sin embargo, esto no implica que su promedio deba ser:

```text
rating_mean = 0
```

En realidad:

```text
rating_mean = NaN
```

porque no existe evidencia suficiente para calcular una media.

Asignar automáticamente un valor de cero introduciría información artificial y generaría interpretaciones incorrectas sobre la percepción de los usuarios.

Este mismo principio se aplicará a cualquier otra métrica derivada de observaciones inexistentes.

## Beneficios de la estrategia de agregación

La construcción previa de métricas a nivel película aporta múltiples ventajas:

- Preserva la granularidad del catálogo.
- Evita uniones many-to-many accidentales.
- Reduce significativamente el volumen de datos durante las integraciones.
- Facilita el análisis descriptivo posterior.
- Mantiene la trazabilidad de las transformaciones realizadas.
- Genera indicadores inmediatamente utilizables para visualización y análisis.

## Conclusiones

La generación de agregados constituye un paso intermedio fundamental entre los datos transaccionales y la construcción de estructuras analíticas más complejas.

Como resultado de esta etapa:

- Se resume la actividad de millones de eventos a nivel película.
- Se generan métricas descriptivas de calificaciones y etiquetas.
- Se preserva la integridad relacional del modelo.
- Se garantiza la unicidad de `movieId` en las tablas agregadas.
- Se prepara la información necesaria para enriquecer el catálogo sin alterar su granularidad original.

Estas nuevas tablas servirán como componentes centrales durante la construcción del catálogo enriquecido y de los conjuntos de datos analíticos que se desarrollarán en las siguientes etapas del proyecto.

## Construcción de la dimensión consolidada de películas

Tras completar la validación de calidad de los datos, normalizar la información de géneros y generar métricas agregadas a partir de las tablas transaccionales, se procede a construir una dimensión consolidada de películas.

El objetivo de esta etapa consiste en integrar en una única estructura toda la información relevante asociada a cada película, preservando en todo momento la granularidad original del catálogo.

La tabla resultante, denominada `movies_consolidated`, se convertirá en uno de los principales activos analíticos del proyecto, ya que permitirá estudiar características descriptivas, actividad de usuarios y métricas derivadas sin necesidad de realizar uniones complejas sobre millones de registros transaccionales.

## Objetivo de la integración

Las tablas analizadas hasta este punto contienen distintos tipos de información:

| Tabla | Información principal |
|---------|---------|
| `movies` | Catálogo de películas |
| `links` | Identificadores externos |
| `rating_agg` | Métricas agregadas de calificaciones |
| `tag_agg` | Métricas agregadas de etiquetas |

Cada una aporta una perspectiva diferente sobre una misma entidad: la película.

La integración busca reunir esta información en una estructura común donde cada observación continúe representando exactamente una película.

## Definición del universo de referencia

Un aspecto crítico de esta etapa consiste en determinar cuál de las tablas actuará como universo de referencia durante la integración.

En este proyecto se adopta como tabla principal:

```text
movies
```

porque representa el catálogo oficial distribuido por MovieLens.

Esto implica que todas las películas presentes en el catálogo deben conservarse independientemente de si poseen actividad registrada o no.

### Justificación

Existen películas que:

- No han recibido calificaciones.
- No han sido etiquetadas.
- Carecen parcialmente de identificadores externos.

Sin embargo, dichas películas siguen formando parte legítima del catálogo.

Por esta razón, eliminar registros únicamente por ausencia de actividad generaría una representación incompleta del universo de películas disponible.

## Estrategia de integración

La construcción de `movies_consolidated` se realiza utilizando un conjunto de uniones tipo:

```text
LEFT JOIN
```

tomando como punto de partida la tabla `movies`.

Este enfoque garantiza que:

- Todas las películas del catálogo permanezcan en la tabla final.
- Ninguna película sea eliminada por falta de actividad.
- Las métricas agregadas se incorporen únicamente cuando exista información disponible.
- La granularidad original se conserve durante todo el proceso.

## Validación explícita de relaciones

Cada integración utiliza el parámetro:

```text
validate="one_to_one"
```

que obliga a Pandas a comprobar que ambas tablas cumplen una relación uno a uno sobre la llave utilizada.

Esta validación constituye una protección adicional contra errores metodológicos frecuentes, tales como:

- Uniones many-to-many accidentales.
- Duplicación de registros.
- Multiplicación no deseada del catálogo.
- Alteración inadvertida de la granularidad de análisis.

Si alguna de las tablas incumpliera esta condición, el proceso se interrumpiría automáticamente.

## Tratamiento de métricas derivadas

Una vez completada la integración, algunas películas presentan valores faltantes en las métricas agregadas debido a que no registran actividad de usuarios.

Es importante distinguir dos tipos de variables:

### Métricas de conteo

Las variables:

- `rating_count`
- `rating_users`
- `tag_application_count`
- `non_null_tag_count`
- `distinct_tags`
- `tag_users`

representan recuentos observables.

Cuando una película no registra actividad, el valor correcto es:

```text
0
```

Por esta razón, los valores faltantes asociados a estas variables se sustituyen explícitamente por cero.

### Métricas estadísticas

Indicadores como:

- `rating_mean`
- `rating_std`
- `rating_min`
- `rating_max`

requieren la existencia de observaciones para ser calculados.

En ausencia de calificaciones, estos valores permanecen como:

```text
NaN
```

ya que no existe evidencia suficiente para estimarlos.

### Principio metodológico

La ausencia de calificaciones no implica una valoración de cero.

Por ejemplo:

```text
rating_count = 0
```

no significa que:

```text
rating_mean = 0
```

sino que el promedio es desconocido debido a la falta de observaciones.

Conservar los nulos evita introducir información artificial en el proceso analítico.

## Construcción de variables indicadoras

Además de las métricas agregadas, se generan indicadores booleanos que facilitan análisis posteriores.

### Disponibilidad de calificaciones

```text
has_ratings
```

Indica si la película posee al menos una calificación registrada.

### Disponibilidad de etiquetas

```text
has_tags
```

Indica si la película ha recibido alguna etiqueta.

### Disponibilidad de identificadores externos

```text
has_imdb_id
```

Permite identificar películas con correspondencia en IMDb.

```text
has_tmdb_id
```

Permite identificar películas con correspondencia en TMDb.

Estas variables simplifican futuros análisis de cobertura y permiten construir filtros sin recalcular condiciones complejas.

## Verificación de la integridad de la integración

Tras finalizar el proceso se realiza una auditoría para asegurar que la estructura del catálogo no haya sido alterada.

Se evalúan cuatro condiciones fundamentales.

### Conservación del número de filas

La cantidad de registros en la tabla consolidada debe coincidir exactamente con el número de películas presentes en el catálogo original.

Resultado observado:

```text
movies_original_rows = 87,585
consolidated_rows = 87,585
```

### Conservación de unicidad

El identificador principal debe seguir siendo único.

Resultado:

```text
consolidated_movieId_unique = True
```

### Preservación completa del catálogo

No debe perderse ninguna película durante la integración.

Resultado:

```text
rows_preserved = True
```

### Ausencia de alteraciones de granularidad

La combinación de validaciones anteriores confirma que la tabla consolidada mantiene exactamente una fila por película.

Esto garantiza que las futuras operaciones de análisis podrán realizarse sobre una dimensión estable y consistente.

In [15]:
movies_base = movies.drop(columns=["title_norm"], errors="ignore").copy()
movies_base["year"]=(movies_base["title"]
                   .str.extract(r"\((\d{4})\)\s*$")[0]
                   .astype("Int64"))
movies_base["title_clean"] = (movies_base["title"]
                            .str.replace(r"\s*\(\d{4}\)\s*$", "", regex=True)
                            .str.strip())
movies_base= movies_base[["movieId", "title_clean", "year", "genres"]]

movies_consolidated = (
    movies_base
    .merge(links, on="movieId", how="left", validate="one_to_one")
    .merge(rating_agg, on="movieId", how="left", validate="one_to_one")
    .merge(tag_agg, on="movieId", how="left", validate="one_to_one")
)

count_columns = [
    "rating_count", "rating_users",
    "tag_application_count", "non_null_tag_count",
    "distinct_tags", "tag_users"
]
for col in count_columns:
    movies_consolidated[col] = movies_consolidated[col].fillna(0).astype("int64")

movies_consolidated["has_ratings"] = movies_consolidated["rating_count"].gt(0)
movies_consolidated["has_tags"] = movies_consolidated["tag_application_count"].gt(0)
movies_consolidated["has_imdb_id"] = movies_consolidated["imdbId"].notna()
movies_consolidated["has_tmdb_id"] = movies_consolidated["tmdbId"].notna()

integration_check = {
    "movies_original_rows": len(movies),
    "consolidated_rows": len(movies_consolidated),
    "consolidated_movieId_unique": bool(movies_consolidated["movieId"].is_unique),
    "rows_preserved": len(movies) == len(movies_consolidated)
}
display(pd.Series(integration_check, name="result"))

if not integration_check["rows_preserved"] or not integration_check["consolidated_movieId_unique"]:
    raise AssertionError("La integración alteró la granularidad del catálogo.")

movies_original_rows           87585
consolidated_rows              87585
consolidated_movieId_unique     True
rows_preserved                  True
Name: result, dtype: object

In [19]:
movies_consolidated.head()

,movieId,title_clean,year,genres,imdbId,tmdbId,rating_count,rating_mean,rating_std,rating_min,rating_max,rating_users,tag_application_count,non_null_tag_count,distinct_tags,tag_users,has_ratings,has_tags,has_imdb_id,has_tmdb_id
0,1,Toy Story,1995,Adventure|Animation|Children|Comedy|Fantasy,114709,862,68997,3.8974,0.9215,0.5000,5.0000,68997,1230,1230,435,255,True,True,True,True
1,2,Jumanji,1995,Adventure|Children|Fantasy,113497,8844,28904,3.2758,0.9555,0.5000,5.0000,28904,573,573,348,83,True,True,True,True
2,3,Grumpier Old Men,1995,Comedy|Romance,113228,15602,13134,3.1394,1.0123,0.5000,5.0000,13134,23,23,23,15,True,True,True,True
3,4,Waiting to Exhale,1995,Comedy|Drama|Romance,114885,31357,2806,2.8453,1.1059,0.5000,5.0000,2806,12,12,9,6,True,True,True,True
4,5,Father of the Bride Part II,1995,Comedy,113041,11862,13154,3.0596,0.9991,0.5000,5.0000,13154,64,64,35,14,True,True,True,True


## Resultado obtenido

La tabla `movies_consolidated` constituye ahora una vista enriquecida del catálogo MovieLens que integra:

- Información descriptiva de las películas.
- Identificadores externos.
- Métricas de calificaciones.
- Métricas de etiquetado.
- Indicadores de disponibilidad de actividad.

Todo ello preservando la estructura:

```text
1 película = 1 fila
```

que constituye el principio fundamental de esta dimensión.

## Conclusiones

La construcción de `movies_consolidated` representa uno de los hitos principales de la fase de integración de datos.

Los resultados obtenidos demuestran que:

- La integración preservó completamente el catálogo original.
- No se introdujeron duplicaciones de películas.
- Se mantuvo la unicidad de `movieId`.
- Las métricas agregadas se incorporaron de forma segura.
- Los conteos se diferenciaron correctamente de las métricas estadísticas.
- La información quedó preparada para análisis exploratorios y comparativos posteriores.

Con esta nueva estructura se dispone de una dimensión analítica robusta que permitirá estudiar el comportamiento de las películas desde múltiples perspectivas sin depender directamente de los millones de registros presentes en las tablas transaccionales originales.

## Construcción de tablas de eventos enriquecidas y estrategia de almacenamiento analítico

Hasta este punto del proceso se han construido dos tipos de estructuras complementarias:

- Una **dimensión consolidada de películas** (`movies_consolidated`), donde existe exactamente una fila por película.
- Un conjunto de **tablas transaccionales** (`ratings` y `tags`) que conservan el detalle completo de las interacciones realizadas por los usuarios.

Aunque la dimensión consolidada resulta ideal para análisis agregados a nivel de película, existen numerosos escenarios analíticos donde es necesario conservar el nivel de detalle original de cada evento.

Por ejemplo:

- Analizar la evolución temporal de las calificaciones.
- Estudiar el comportamiento individual de los usuarios.
- Examinar patrones de etiquetado.
- Construir modelos de recomendación.
- Analizar secuencias de interacción entre usuarios y películas.

Para estos casos resulta conveniente disponer de tablas enriquecidas que combinen los eventos originales con la información descriptiva de cada película.

## Objetivo de las tablas enriquecidas

El propósito de esta etapa consiste en incorporar información contextual a los eventos registrados en las tablas transaccionales.

Se generan dos nuevas estructuras:

### `ratings_enriched`

Conserva la granularidad original de una fila por calificación.

Cada registro mantiene:

- Identificador de usuario.
- Identificador de película.
- Calificación otorgada.
- Marca temporal original.

y añade:

- Título de la película.
- Géneros asociados.

### `tags_enriched`

Conserva la granularidad original de una fila por aplicación de etiqueta.

Cada registro mantiene:

- Identificador de usuario.
- Identificador de película.
- Texto de la etiqueta.
- Marca temporal original.

y añade:

- Título de la película.
- Géneros asociados.

De esta forma, los análisis posteriores pueden realizarse sin necesidad de consultar continuamente la dimensión de películas.

## Principio de separación entre eventos

Un aspecto metodológico importante consiste en mantener separadas las tablas de ratings y tags.

Aunque ambas describen actividad de usuarios sobre películas, representan fenómenos diferentes y poseen granularidades propias.

### Riesgo de una integración directa

Unir directamente las tablas `ratings` y `tags` utilizando únicamente `movieId` produciría una expansión artificial del número de registros.

Por ejemplo:

- Una película con 1,000 calificaciones.
- Una película con 50 etiquetas.

Generaría potencialmente:

```text
1,000 × 50 = 50,000 filas
```

tras una unión many-to-many.

Este fenómeno no representa información nueva y únicamente introduce duplicaciones derivadas del proceso de integración.

Por esta razón:

> Las tablas `ratings_enriched` y `tags_enriched` se construyen y almacenan por separado.

## Construcción de una dimensión mínima de consulta

Para enriquecer las tablas transaccionales no se utiliza la tabla consolidada completa.

En su lugar se genera una dimensión mínima que contiene únicamente:

- `movieId`
- `title`
- `genres`

Esta estrategia ofrece varias ventajas:

- Reduce el consumo de memoria.
- Simplifica las operaciones de integración.
- Evita incorporar columnas innecesarias.
- Mejora el rendimiento durante el procesamiento de grandes volúmenes de datos.

La dimensión funciona como una tabla de consulta ligera utilizada exclusivamente para enriquecer los eventos.

## Procesamiento por bloques

Uno de los principales desafíos del conjunto MovieLens 32M es el tamaño de la tabla de calificaciones.

Con más de treinta millones de registros, realizar una integración completa en memoria podría:

- Incrementar significativamente el consumo de RAM.
- Reducir el rendimiento.
- Limitar la reproducibilidad en equipos con recursos moderados.

Por esta razón se adopta una estrategia de procesamiento incremental basada en bloques (`chunks`).

### Funcionamiento del procesamiento por bloques

Los datos se leen en fragmentos de tamaño controlado.

Cada bloque:

1. Se carga desde el archivo original.
2. Se integra con la dimensión de películas.
3. Se transforma.
4. Se escribe inmediatamente al almacenamiento de salida.

Posteriormente el bloque es liberado de memoria y el proceso continúa con el siguiente fragmento.

Este enfoque permite procesar conjuntos de datos muy grandes sin necesidad de mantener una copia completa del dataset enriquecido en memoria.

## Incorporación de variables temporales

La documentación original distribuye las marcas de tiempo en formato Unix.

Por motivos de trazabilidad se conserva la columna:

```text
timestamp
```

exactamente como aparece en los archivos fuente.

Sin embargo, para facilitar futuros análisis temporales, se añade una segunda representación:

```text
timestamp_utc
```

obtenida mediante la conversión del timestamp Unix a un objeto datetime en UTC.

Esta estrategia ofrece dos ventajas importantes:

### Conservación del dato original

Se mantiene intacta la información distribuida por MovieLens.

### Facilidad de análisis

Los procesos posteriores pueden trabajar directamente con fechas interpretables sin necesidad de repetir conversiones.

## Estrategia de almacenamiento

Los conjuntos enriquecidos pueden alcanzar tamaños considerablemente mayores que las tablas originales debido a la incorporación de información descriptiva adicional.

Por esta razón, el sistema prioriza el uso de formatos analíticos más eficientes.

### Uso preferente de Parquet

Cuando existe un motor compatible disponible (`pyarrow` o `fastparquet`), los datos se almacenan utilizando formato:

```text
Parquet
```

Parquet proporciona diversas ventajas para análisis de datos:

- Menor tamaño de almacenamiento.
- Lectura más rápida.
- Compresión eficiente.
- Optimización para procesamiento columnar.
- Integración nativa con Pandas y herramientas de ciencia de datos.

Además, cada bloque procesado puede almacenarse como un fragmento independiente dentro de un dataset Parquet, evitando la necesidad de generar un archivo monolítico de gran tamaño.

### Alternativa CSV

Si no existe un motor Parquet disponible en el entorno de ejecución, el procedimiento utiliza CSV como mecanismo alternativo.

En este caso:

- El archivo se construye progresivamente.
- El encabezado se escribe una única vez.
- Los bloques posteriores se anexan secuencialmente.

Aunque menos eficiente que Parquet, esta estrategia garantiza la portabilidad y reproducibilidad del proyecto.

In [16]:
# Dimensión mínima de película para enriquecer tablas de eventos.
movie_lookup = movies[["movieId", "title", "genres"]].copy()

def write_enriched_in_chunks(input_path, output_stem, table_kind, chunksize=500_000):
    """Enriquece una tabla de eventos por bloques sin cargar otra copia completa."""
    output_parquet = PROCESSED_DIR / f"{output_stem}.parquet"
    output_csv = PROCESSED_DIR / f"{output_stem}.csv"

    # Preferir Parquet si existe un motor compatible.
    try:
        import pyarrow  # noqa: F401
        parquet_available = True
    except ImportError:
        try:
            import fastparquet  # noqa: F401
            parquet_available = True
        except ImportError:
            parquet_available = False

    if parquet_available:
        # Escribir fragmentos Parquet independientes y después combinarlos como dataset.
        # Esta estrategia evita requerir un único archivo Parquet monolítico.
        fragment_dir = PROCESSED_DIR / f"{output_stem}_parquet"
        fragment_dir.mkdir(parents=True, exist_ok=True)
        for old_file in fragment_dir.glob("part-*.parquet"):
            old_file.unlink()

        for i, chunk in enumerate(pd.read_csv(input_path, chunksize=chunksize)):
            chunk = chunk.merge(movie_lookup, on="movieId", how="left", validate="many_to_one")
            chunk["timestamp_utc"] = pd.to_datetime(
                chunk["timestamp"], unit="s", utc=True, errors="coerce"
            )
            chunk.to_parquet(
                fragment_dir / f"part-{i:05d}.parquet",
                index=False
            )
        return fragment_dir

    # Alternativa CSV: escribir encabezado una sola vez.
    if output_csv.exists():
        output_csv.unlink()

    for i, chunk in enumerate(pd.read_csv(input_path, chunksize=chunksize)):
        chunk = chunk.merge(movie_lookup, on="movieId", how="left", validate="many_to_one")
        chunk["timestamp_utc"] = pd.to_datetime(
            chunk["timestamp"], unit="s", utc=True, errors="coerce"
        )
        chunk.to_csv(
            output_csv,
            mode="w" if i == 0 else "a",
            header=(i == 0),
            index=False,
            encoding="utf-8"
        )
    return output_csv

ratings_enriched_path = write_enriched_in_chunks(
    FILES["ratings"], "ratings_enriched", "ratings"
)
tags_enriched_path = write_enriched_in_chunks(
    FILES["tags"], "tags_enriched", "tags"
)

print("Ratings enriquecidos:", ratings_enriched_path)
print("Tags enriquecidos:", tags_enriched_path)

Ratings enriquecidos: c:\Users\igpea\OneDrive\Documents\Diplomado\Proyecto_Integrador_M1\data\processed\ratings_enriched.csv
Tags enriquecidos: c:\Users\igpea\OneDrive\Documents\Diplomado\Proyecto_Integrador_M1\data\processed\tags_enriched.csv


## Resultados obtenidos

La ejecución del procedimiento genera satisfactoriamente:

```text
ratings_enriched
```

y

```text
tags_enriched
```

dentro del directorio:

```text
data/processed/
```

En el entorno actual, ambos conjuntos fueron almacenados como archivos CSV:

```text
ratings_enriched.csv
tags_enriched.csv
```

quedando disponibles para análisis posteriores.

## Beneficios de la estrategia implementada

La construcción de tablas de eventos enriquecidas aporta múltiples ventajas analíticas y operativas:

- Preserva la granularidad original de los eventos.
- Evita uniones many-to-many accidentales.
- Facilita consultas descriptivas posteriores.
- Incorpora contexto de película directamente en los registros.
- Conserva las marcas temporales originales.
- Añade representaciones temporales listas para análisis.
- Permite trabajar con volúmenes masivos de datos mediante procesamiento por bloques.
- Optimiza el almacenamiento mediante formatos analíticos cuando están disponibles.

## Conclusiones

Con esta etapa se completa la construcción de los principales activos analíticos derivados del conjunto MovieLens.

A partir de este momento el proyecto dispone de:

- Una dimensión consolidada de películas (`movies_consolidated`).
- Una tabla puente normalizada película–género (`movie_genres`).
- Una tabla enriquecida de calificaciones (`ratings_enriched`).
- Una tabla enriquecida de etiquetas (`tags_enriched`).

Estas estructuras conservan la integridad del modelo original, preservan las distintas granularidades de análisis y proporcionan una base sólida para las etapas posteriores de exploración visual, análisis estadístico y construcción de modelos analíticos.

## Validaciones posteriores a la limpieza e integración

Una vez completadas las etapas de limpieza, normalización e integración de datos, resulta imprescindible verificar que las transformaciones realizadas no hayan alterado involuntariamente la estructura original del modelo de datos.

Esta fase funciona como una etapa formal de aseguramiento de calidad y tiene como propósito comprobar que los productos generados conservan las propiedades esperadas de unicidad, integridad referencial y consistencia numérica.

A diferencia de las validaciones exploratorias realizadas anteriormente, estas comprobaciones actúan como controles de aceptación del proceso. Si alguna condición crítica no se cumple, la ejecución del notebook se interrumpe automáticamente para evitar que errores estructurales se propaguen hacia etapas posteriores de análisis.

## Objetivo de las validaciones finales

Durante las fases de limpieza e integración se generaron nuevas estructuras derivadas:

- La dimensión consolidada de películas (`movies_consolidated`).
- La tabla puente película–género (`movie_genres`).
- Los agregados de calificaciones (`rating_agg`).
- Los agregados de etiquetas (`tag_agg`).

Cada una de estas estructuras debe cumplir ciertas reglas de negocio y consistencia derivadas del diseño original del dataset.

Las validaciones finales tienen como objetivo comprobar que:

- No se perdieron registros durante la integración.
- No se introdujeron duplicados.
- Las relaciones siguen siendo válidas.
- Los agregados representan exactamente los datos originales.
- Se preservó la granularidad prevista para cada tabla.

## Verificación de la dimensión consolidada de películas

La tabla `movies_consolidated` constituye el principal producto de integración generado durante el proceso.

Como se definió previamente, esta estructura debe mantener una relación:

```text
1 película = 1 fila
```

Para comprobarlo se evalúan dos condiciones fundamentales.

### Unicidad de `movieId`

El identificador de película debe seguir siendo único después de integrar información procedente de múltiples fuentes.

Esta validación permite detectar:

- Duplicaciones introducidas por errores de integración.
- Relaciones many-to-many accidentales.
- Expansiones no deseadas del catálogo.

Resultado:

```text
movieId único en movies_consolidated → Correcto
```

### Conservación del número de películas

La cantidad total de registros en la tabla consolidada debe coincidir exactamente con la cantidad de películas presentes en el catálogo original.

Esto debe confirmar que ninguna película fue eliminada ni duplicada durante el proceso de integración.

## Validación de la tabla puente película–género

La tabla `movie_genres` constituye una representación normalizada de la relación entre películas y géneros.

Debido a su naturaleza relacional, resulta necesario verificar dos propiedades esenciales.

### Unicidad de la combinación `(movieId, genre)`

Cada combinación película–género debe aparecer una sola vez.

La existencia de duplicados en esta relación produciría:

- Sobreestimación de frecuencias.
- Distorsiones en agregaciones posteriores.
- Errores en análisis de distribución por género.

Resultado:

```text
Combinación movieId–genre única → Correcto
```

### Integridad referencial de géneros

Toda película presente en la tabla puente debe existir previamente dentro del catálogo principal.

Para comprobarlo se verifica que todos los `movieId` registrados en `movie_genres` pertenezcan al conjunto de películas válidas del catálogo.

Resultado:

```text
Todos los géneros están vinculados a películas existentes
```

No se detectan identificadores huérfanos.

## Validación de los agregados de actividad

Las tablas agregadas representan resúmenes construidos a partir de millones de registros transaccionales.

Por esta razón resulta fundamental confirmar que:

- Mantienen una fila por película.
- Conservan correctamente la información resumida.
- No se han perdido observaciones durante el proceso de agrupación.

### Unicidad de los agregados

Las tablas:

- `rating_agg`
- `tag_agg`

deben contener una única observación por `movieId`.

Esta propiedad garantiza que puedan integrarse posteriormente mediante relaciones uno a uno.

La validación confirma que ambas tablas cumplen esta condición.

## Reconciliación de conteos

Una de las verificaciones más importantes consiste en comprobar que las métricas agregadas reconstruyan exactamente el volumen original de eventos.

### Reconciliación de calificaciones

La suma de:

```text
rating_count
```

debe ser exactamente igual al número total de filas presentes en la tabla `ratings`.

Conceptualmente:

```text
Σ rating_count = total de calificaciones originales
```

Resultado:

```text
Agregado de ratings reconcilia con ratings original → Correcto
```

Esto demuestra que ninguna calificación fue omitida o duplicada durante el procedimiento de agregación.

### Reconciliación de etiquetas

De forma análoga, la suma de:

```text
tag_application_count
```

debe coincidir con el número total de registros de la tabla `tags`.

Resultado:

```text
Agregado de tags reconcilia con tags original → Correcto
```

Por tanto, todas las aplicaciones de etiquetas fueron incorporadas correctamente a la tabla agregada.

## Estrategia de control mediante afirmaciones (`assert`)

Las verificaciones críticas se implementan utilizando instrucciones `assert`.

Este mecanismo permite transformar las reglas metodológicas del proyecto en condiciones obligatorias de ejecución.

Si cualquiera de estas condiciones falla, el notebook detiene automáticamente su ejecución y genera una excepción.

Entre las ventajas de este enfoque se encuentran:

- Detección temprana de errores.
- Prevención de resultados inconsistentes.
- Automatización de controles de calidad.
- Mayor reproducibilidad del proceso.

De este modo, la validez de los datos no depende únicamente de inspecciones visuales sino también de comprobaciones programáticas verificables.

In [17]:
assert movies_consolidated["movieId"].is_unique
assert len(movies_consolidated) == len(movies)
assert not movie_genres.duplicated(["movieId", "genre"]).any()
assert set(movie_genres["movieId"]).issubset(catalog_movie_ids)
assert rating_agg["movieId"].is_unique
assert tag_agg["movieId"].is_unique

# Reconciliación de conteos.
assert int(rating_agg["rating_count"].sum()) == len(ratings)
assert int(tag_agg["tag_application_count"].sum()) == len(tags)

post_validation = pd.DataFrame([
    {"test": "movieId único en movies_consolidated",
     "passed": movies_consolidated["movieId"].is_unique},
    {"test": "Se preservó el total de películas",
     "passed": len(movies_consolidated) == len(movies)},
    {"test": "Combinación movieId-genre única",
     "passed": not movie_genres.duplicated(["movieId", "genre"]).any()},
    {"test": "Géneros vinculados a películas existentes",
     "passed": set(movie_genres["movieId"]).issubset(catalog_movie_ids)},
    {"test": "Agregado de ratings reconcilia con ratings original",
     "passed": int(rating_agg["rating_count"].sum()) == len(ratings)},
    {"test": "Agregado de tags reconcilia con tags original",
     "passed": int(tag_agg["tag_application_count"].sum()) == len(tags)}
])
display(post_validation)

if not post_validation["passed"].all():
    raise AssertionError("Una o más validaciones posteriores fallaron.")

,test,passed
0,movieId único en movies_consolidated,True
1,Se preservó el total de películas,True
2,Combinación movieId-genre única,True
3,Géneros vinculados a películas existentes,True
4,Agregado de ratings reconcilia con ratings ori...,True
5,Agregado de tags reconcilia con tags original,True


## Resultados obtenidos

Las seis validaciones ejecutadas fueron superadas satisfactoriamente. No se identificó ninguna inconsistencia estructural en los datos procesados.

## Conclusiones

Las validaciones posteriores confirman que el proceso de limpieza e integración preservó correctamente las propiedades fundamentales del modelo de datos.

Los principales hallazgos son:

- La granularidad de la dimensión de películas permanece intacta.
- No se introdujeron duplicaciones durante las integraciones.
- La tabla puente mantiene integridad referencial completa.
- Los agregados reproducen exactamente el volumen de eventos originales.
- No se detectaron pérdidas de información durante las transformaciones.

En consecuencia, las estructuras generadas pueden considerarse productos analíticos válidos y confiables para las siguientes etapas del proyecto, incluyendo exploración visual, análisis descriptivo avanzado y generación de modelos analíticos.

## Exportación de resultados y generación de artefactos reproducibles

Una vez completadas las etapas de validación, normalización, agregación e integración de datos, se procede a materializar los productos generados mediante la exportación de los resultados a la capa de datos procesados del proyecto.

Esta etapa marca la transición entre el proceso de preparación de datos y las futuras fases de exploración, visualización y análisis estadístico.

El objetivo principal es almacenar de forma estructurada todos los artefactos analíticos generados, permitiendo que puedan ser reutilizados posteriormente sin necesidad de repetir cada transformación intermedia.

## Principios de almacenamiento adoptados

La estrategia de exportación sigue tres principios fundamentales:

### Reproducibilidad

Todos los archivos exportados son productos derivados que pueden reconstruirse ejecutando nuevamente el notebook.

No se generan transformaciones manuales ni intervenciones externas sobre los archivos resultantes.

### Trazabilidad

Cada archivo exportado responde a una finalidad analítica específica y conserva una relación directa con alguna de las etapas desarrolladas durante el proceso de limpieza e integración.

Esto facilita auditar posteriormente cómo se generó cada resultado.

### Separación entre datos originales y datos procesados

Los archivos originales distribuidos por MovieLens permanecen intactos dentro de:

```text
data/raw/
```

mientras que todos los productos generados por el proyecto se almacenan en:

```text
data/processed/
```

Esta separación evita modificaciones accidentales sobre las fuentes originales de información y permite reconstruir completamente cualquier etapa del procesamiento.

## Exportación de estructuras analíticas principales

### Catálogo consolidado de películas

```text
movies_consolidated.csv
```

Constituye el principal producto analítico generado durante esta fase.

Posee exactamente una fila por película e integra:

- Información descriptiva.
- Identificadores externos.
- Métricas agregadas de calificaciones.
- Métricas agregadas de etiquetas.
- Indicadores de disponibilidad de actividad.

Su propósito es servir como dimensión enriquecida para análisis posteriores a nivel de película.

---

### Tabla puente película–género

```text
movie_genres.csv
```

Contiene una fila por combinación:

```text
movieId – genre
```

Esta estructura normalizada permite realizar análisis por género sin necesidad de procesar nuevamente la columna multivaluada `genres`.

Entre sus posibles aplicaciones se encuentran:

- Distribuciones por género.
- Comparaciones entre categorías cinematográficas.
- Métricas agregadas segmentadas por género.

---

### Métricas agregadas de calificaciones

```text
rating_aggregates_by_movie.csv
```

Resume la actividad de calificación observada para cada película.

Incluye indicadores como:

- Número de calificaciones.
- Promedio.
- Desviación estándar.
- Valores mínimos y máximos.
- Número de usuarios calificadores.

Esta tabla permite realizar análisis descriptivos sin necesidad de consultar los millones de registros contenidos en `ratings`.

---

### Métricas agregadas de etiquetas

```text
tag_aggregates_by_movie.csv
```

Contiene indicadores derivados de la actividad de etiquetado.

Entre otras métricas se incluyen:

- Número total de aplicaciones de etiquetas.
- Número de etiquetas distintas.
- Número de usuarios etiquetadores.

Esta información complementa los indicadores de interacción de usuarios sobre las películas.

## Exportación de diagnósticos de cobertura

### Películas sin calificaciones

```text
movies_without_ratings.csv
```

Incluye las películas pertenecientes al catálogo que no presentan actividad de calificación.

El archivo puede utilizarse para:

- Evaluar cobertura del catálogo.
- Analizar sesgos de participación.
- Identificar películas con baja interacción.

---

### Películas sin etiquetas

```text
movies_without_tags.csv
```

Contiene las películas que no poseen registros de etiquetado.

Permite estudiar diferencias entre películas frecuentemente descritas por los usuarios y aquellas que permanecen sin anotaciones semánticas.

## Exportación de diagnósticos de calidad

### Resumen general de calidad

```text
quality_overview.csv
```

Almacena los indicadores globales calculados para cada tabla:

- Número de registros.
- Número de columnas.
- Valores nulos.
- Duplicados.
- Completitud.
- Consumo de memoria.

Constituye una referencia rápida del estado general de los datos.

---

### Perfil de calidad por columna

```text
column_quality.csv
```

Contiene métricas individuales para cada variable:

- Tipo de dato.
- Valores nulos.
- Porcentaje de completitud.
- Cardinalidad.
- Conteos de observaciones válidas.

Este archivo facilita futuras auditorías de calidad.

---

### Integridad referencial

```text
referential_integrity.csv
```

Documenta las verificaciones realizadas sobre las correspondencias de `movieId` entre las distintas tablas del modelo.

Permite demostrar la consistencia relacional observada durante el proceso de integración.

## Exportación de diagnósticos complementarios

Además de los productos principales, se almacenan diversos resultados generados durante la fase de validación.

Entre ellos destacan:

### Validación de llaves

```text
key_checks.csv
```

Resume las comprobaciones realizadas sobre identificadores y llaves candidatas.

---

### Diagnóstico de títulos repetidos

```text
title_duplicates_exact.csv
```

Documenta títulos repetidos detectados durante la exploración del catálogo.

---

### Distribución de géneros

```text
genre_frequency.csv
```

Contiene la frecuencia observada de cada género cinematográfico dentro del catálogo.

---

### Distribución de calificaciones

```text
rating_distribution.csv
```

Registra las frecuencias y proporciones de cada valor de la escala de calificación.

---

### Resultados de validación final

```text
post_validation.csv
```

Almacena las comprobaciones de aceptación realizadas al concluir el proceso de integración.

## Exportación de diagnósticos especializados

Algunos resultados se almacenan dentro del directorio:

```text
data/processed/diagnostics/
```

con el objetivo de separar los productos analíticos de los artefactos de control de calidad.

### Reporte de valores nulos

```text
null_report.csv
```

Contiene el detalle de las variables que presentan valores faltantes.

---

### Películas sin identificador TMDb

```text
links_without_tmdb.csv
```

Documenta los registros donde la correspondencia con TMDb no se encuentra disponible.

---

### Etiquetas sin contenido textual

```text
tags_without_text.csv
```

Almacena los registros de etiquetado donde el atributo `tag` presenta valores nulos.

La conservación de estos registros facilita futuras auditorías sin alterar las tablas base.

## Tablas enriquecidas de eventos

Las estructuras:

```text
ratings_enriched
```

y

```text
tags_enriched
```

también forman parte de los productos generados durante esta fase.

Dependiendo de las bibliotecas disponibles en el entorno, estas tablas pueden almacenarse mediante dos estrategias:

### Formato Parquet

Cuando existe soporte para:

- `pyarrow`
- `fastparquet`

los datos se exportan como datasets Parquet particionados.

Este formato ofrece importantes ventajas:

- Menor tamaño de almacenamiento.
- Mayor velocidad de lectura.
- Mejor rendimiento analítico.

### Formato CSV

Cuando no existe un motor Parquet disponible, los datos se exportan como archivos CSV.

Aunque menos eficiente, esta alternativa garantiza portabilidad y compatibilidad universal.

## Resultado final de la etapa

La ejecución satisfactoria de esta fase confirma que todos los productos derivados fueron almacenados correctamente dentro del directorio:

```text
data/processed/
```

A partir de este momento el proyecto dispone de un conjunto completo de:

- Tablas consolidadas.
- Tablas normalizadas.
- Métricas agregadas.
- Diagnósticos de calidad.
- Validaciones estructurales.
- Eventos enriquecidos.

Todos ellos reproducibles, documentados y listos para ser utilizados durante las siguientes etapas de exploración visual, análisis estadístico y generación de conocimiento sobre el comportamiento de usuarios y películas dentro del ecosistema MovieLens.

In [18]:
# Exportar tablas principales.
exports = {
    "movies_consolidated.csv": movies_consolidated,
    "movie_genres.csv": movie_genres_imp,
    "rating_aggregates_by_movie.csv": rating_agg,
    "tag_aggregates_by_movie.csv": tag_agg,
    "movies_without_ratings.csv": movies_without_ratings,
    "movies_without_tags.csv": movies_without_tags,
    "quality_overview.csv": overview,
    "column_quality.csv": column_quality,
    "referential_integrity.csv": referential_report,
    "key_checks.csv": key_checks,
    "title_duplicates_exact.csv": title_counts,
    "genre_frequency.csv": genre_frequency,
    "rating_distribution.csv": rating_distribution,
    "post_validation.csv": post_validation
}

for filename, df in exports.items():
    df.to_csv(PROCESSED_DIR / filename, index=False, encoding="utf-8")

null_report.to_csv(
    DIAGNOSTICS_DIR / "null_report.csv", index=False, encoding="utf-8"
)
tmdb_missing.to_csv(
    DIAGNOSTICS_DIR / "links_without_tmdb.csv", index=False, encoding="utf-8"
)
tags_null.to_csv(
    DIAGNOSTICS_DIR / "tags_without_text.csv", index=False, encoding="utf-8"
)

print(f"Exportación completada en: {PROCESSED_DIR}")

Exportación completada en: c:\Users\igpea\OneDrive\Documents\Diplomado\Proyecto_Integrador_M1\data\processed
